In [13]:
import cv2
import numpy as np
import uiautomator2 as u2
img = u2.connect('7fe98fc6').screenshot(format='opencv')[204:772]

# 1. 讀取與灰階
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 二值化 (反轉顏色，讓線條變白色，背景變黑色)
# 這裡使用 adaptiveThreshold 自動適應光線變化
binary = cv2.adaptiveThreshold(~gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 15, -2)

# 3. 定義一個「水平」的結構元素
# (50, 1) 表示寬度 50，高度 1。這會過濾掉所有寬度小於 50 的東西（例如文字）
horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))

# 4. 形態學操作：侵蝕後膨脹 (開運算) -> 只留下水平長線
detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel)

# ---此時 detected_lines 已經是一張只有水平線的黑白圖了---

# 5. (選用) 如果你想把這些線的座標找出來
contours, hierarchy = cv2.findContours(detected_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    # 畫出紅色的框或線
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 2)

cv2.imshow("Morphological Lines", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# 修正後的停車場管理器測試 - 正確處理檢查次數
from json_manager import create_park_manager

def test_park_manager_with_check_count():
    """測試停車場管理器 - 正確處理檢查次數"""
    print("=== 修正後的停車場管理器測試 ===")
    manager = create_park_manager("test-device-corrected")
    
    # 1. 檢查初始狀態
    print("1. 初始狀態檢查:")
    daily_ts, daily_num, weekly_ts, weekly_num = manager.get_buy_data()
    print(f"   Daily: timestamp={daily_ts}, buy_num={daily_num}")
    print(f"   Weekly: timestamp={weekly_ts}, buy_num={weekly_num}")
    
    # 2. 檢查是否需要購買
    print("\n2. 檢查是否需要購買:")
    should_buy_daily = manager.should_purchase('daily')
    should_buy_weekly = manager.should_purchase('weekly')
    print(f"   應該購買 daily: {should_buy_daily}")
    print(f"   應該購買 weekly: {should_buy_weekly}")
    
    # 3. 記錄購買 - 正確傳遞檢查次數
    print("\n3. 記錄購買 (檢查次數設為1):")
    manager.record_purchase('daily', buy_num=1, check_time=1)  # 檢查次數設為1
    manager.record_purchase('weekly', buy_num=1, check_time=1)  # 檢查次數設為1
    hase('weekly', buy_num=1, check_time=1)  # 檢查次數設為1
    
    # 4. 驗證數據是否正確更新
    print("\n4. 驗證更新後的數據:")
    data = manager.load_data()
    print(f"   Daily 數據: {data.get('daily', {})}")
    print(f"   Weekly 數據: {data.get('weekly', {})}")
    
    # 5. 再次檢查購買狀態
    print("\n5. 重新檢查購買狀態:")
    daily_ts, daily_num, weekly_ts, weekly_num = manager.get_buy_data()
    print(f"   Daily: timestamp={daily_ts}, buy_num={daily_num}")
    print(f"   Weekly: timestamp={weekly_ts}, buy_num={weekly_num}")
    
    should_buy_daily = manager.should_purchase('daily')
    should_buy_weekly = manager.should_purchase('weekly')
    print(f"   現在應該購買 daily: {should_buy_daily}")
    print(f"   現在應該購買 weekly: {should_buy_weekly}")

# 執行測試
test_park_manager_with_check_count()

=== 修正後的停車場管理器測試 ===
1. 初始狀態檢查:
   Daily: timestamp=0.0, buy_num=0
   Weekly: timestamp=0.0, buy_num=0

2. 檢查是否需要購買:
   應該購買 daily: True
   應該購買 weekly: True

3. 記錄購買 (檢查次數設為1):
已記錄 daily 購買數據，數量: 1, 檢查次數: 1
已記錄 weekly 購買數據，數量: 1, 檢查次數: 1

4. 驗證更新後的數據:
   Daily 數據: {'car_market_timestamp': 1759184537.588436, 'car_market_buy_num': 1, 'car_market_check_time': 1}
   Weekly 數據: {'car_market_timestamp': 1759184537.607417, 'car_market_buy_num': 1, 'car_market_check_time': 1}

5. 重新檢查購買狀態:
   Daily: timestamp=1759184537.588436, buy_num=1
   Weekly: timestamp=1759184537.607417, buy_num=1
   現在應該購買 daily: True
   現在應該購買 weekly: True


In [ ]:

import uiautomator2 as u2
import img_tools
import cv2
d=u2.connect('7fe98fc6')
def preprocess_for_ocr(img_roi):
    if img_roi is None or img_roi.size == 0: return None
    gray = cv2.cvtColor(img_roi, cv2.COLOR_BGR2GRAY)
    scaled = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    binary = cv2.adaptiveThreshold(
        scaled, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 25, 10
    )
    return binary
ROI_Y_START = 200
ROI_Y_END = 777
raw_img =d.screenshot(format="opencv")
list_area = raw_img[ROI_Y_START:ROI_Y_END, :]

processed_block = preprocess_for_ocr(list_area)
ocr_result = img_tools.analyze_skill_via_http(processed_block)
print(f"Block Y: {ROI_Y_START} - OCR Result: {ocr_result}")
cv2.imwrite("processed_block.png", processed_block)

Block Y: 200 - OCR Result: {'combo': '', 'is_unwanted': False, 'normalized_combo': '', 'ocr_results': [{'bbox': [295, 77, 748, 118], 'bbox_rel': [0.27314814814814814, 0.06790123456790123, 0.6925925925925925, 0.10405643738977072], 'score': 0.951803982257843, 'text': '[S1288]腦袋被踢成功搶侶'}, {'bbox': [780, 93, 947, 131], 'bbox_rel': [0.7222222222222222, 0.082010582010582, 0.8768518518518519, 0.11552028218694885], 'score': 0.9998733401298523, 'text': '2026/01/29'}, {'bbox': [296, 118, 626, 160], 'bbox_rel': [0.2740740740740741, 0.10405643738977072, 0.5796296296296296, 0.14109347442680775], 'score': 0.9647422432899475, 'text': '[S1467]的跨界車位3'}, {'bbox': [804, 124, 926, 159], 'bbox_rel': [0.7444444444444445, 0.10934744268077601, 0.8574074074074074, 0.1402116402116402], 'score': 0.9824858903884888, 'text': '05:30:30'}, {'bbox': [241, 238, 248, 243], 'bbox_rel': [0.22314814814814815, 0.20987654320987653, 0.22962962962962963, 0.21428571428571427], 'score': 0.3664792776107788, 'text': '「'}, {'bbox':

True

In [ ]:
import cv2
import numpy as np
from datetime import datetime
def preprocess_for_ocr(img_roi):
    if img_roi is None or img_roi.size == 0: return None
    gray = cv2.cvtColor(img_roi, cv2.COLOR_BGR2GRAY)
    scaled = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    binary = cv2.adaptiveThreshold(
        scaled, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 25, 10
    )
    return binary
ROI_Y_START = 200
ROI_Y_END = 777
d = u2.connect('7fe98fc6')
import img_tools
raw_img =d.screenshot(format="opencv")[ROI_Y_START:ROI_Y_END, :]
def cut_img(raw_img):
    gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
    # 1) 二值化（反相，讓線/字變白）
    bw = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV,
        31, 10
)

    # 2) 只保留水平線：水平 kernel 越寬，越能抓長橫線
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (80, 1))
    h_lines = cv2.morphologyEx(bw, cv2.MORPH_OPEN, h_kernel, iterations=1)

    # 3) 把線加粗一點（方便找輪廓）
    h_lines = cv2.dilate(h_lines, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), iterations=1)

    # 4) 找水平線輪廓
    cnts, _ = cv2.findContours(h_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    lines = []
    H, W = bw.shape
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        # 篩選：要夠長、夠扁（你可依畫面調）
        if w > 0.6 * W and h <= 8:
            lines.append((y, x, x+w))

    # 5) 依 y 排序並做去重（同一條線可能被切成多段）
    lines.sort(key=lambda t: t[0])
    ys = []
    for y, x1, x2 in lines:
        if not ys or abs(y - ys[-1]) > 10:   # 10px 內視為同一條
            ys.append(y)

    pad_top = 8
    pad_bottom = 8

    # 例如：切分線之間的區域
    segments = []
    for i in range(len(ys) - 1):
        y1 = max(0, ys[i] + pad_top)
        y2 = min(H, ys[i+1] - pad_bottom)
        if y2 - y1 > 20:
            roi = raw_img[y1:y2, :]
            segments.append(roi)
    return segments

segments = cut_img(raw_img)
print(f"切出 {len(segments)} 個區塊")
# 保存看看

for i, roi in enumerate(segments):
    who_fight_X_start =141
    who_fight_X_end = 387
    what_time_X_start =  391
    what_time_X_end =  481
    # cv2.imshow(f"row_{i:02d}", roi[:, who_fight_X_start:who_fight_X_end])
    # cv2.waitKey(0) 
    who_fight_result = img_tools.analyze_skill_via_http(preprocess_for_ocr(roi[:, who_fight_X_start:who_fight_X_end]))
    what_time_X_result = img_tools.analyze_skill_via_http(preprocess_for_ocr(roi[:, what_time_X_start:what_time_X_end]))
    print("-----")
    who_fight_combined_text = ""
    for r in who_fight_result['ocr_results']:
        who_fight_combined_text += r['text'] + ""
    print(f"Row {i:02d} OCR Result: {who_fight_combined_text}")
    texts = [r['text'] for r in what_time_X_result['ocr_results']]
    raw_str = " ".join(texts)
    try:
    # 按照 OCR 輸出的格式進行匹配 (YYYY/MM/DD HH:MM:SS)
        dt_obj = datetime.strptime(raw_str, "%Y/%m/%d %H:%M:%S")
        print(f"匹配成功！轉換後的物件: {dt_obj}")
        print(f"年份: {dt_obj.year}, 小時: {dt_obj.hour}")
    except ValueError as e:
        print(f"匹配失敗，格式不符: {e}")
cv2.destroyAllWindows()

切出 6 個區塊
-----
Row 00 OCR Result: [S1467]女生都愛我成功搶侶[S1411]的跨界車位1
匹配成功！轉換後的物件: 2026-02-05 04:27:22
年份: 2026, 小時: 4
-----
Row 01 OCR Result: [S1321]小雅成功搶估[S1470]的跨界車位10
匹配成功！轉換後的物件: 2026-02-05 04:20:25
年份: 2026, 小時: 4
-----
Row 02 OCR Result: [S1467]~哇鯊咪~成功搶估[S1470的跨界車位11
匹配成功！轉換後的物件: 2026-02-05 04:09:59
年份: 2026, 小時: 4
-----
Row 03 OCR Result: [S1321]花落成功搶估[S1411]的跨界車位12
匹配成功！轉換後的物件: 2026-02-05 03:51:12
年份: 2026, 小時: 3
-----
Row 04 OCR Result: [S1321]PhD@NTU成功搶估[S1467]的跨界車位8
匹配失敗，格式不符: time data '2026/02/05** 03:21:59' does not match format '%Y/%m/%d %H:%M:%S'
-----
Row 05 OCR Result: ..[S1467]成功搶侶[S1470]的跨界車位9
匹配成功！轉換後的物件: 2026-02-05 03:16:03
年份: 2026, 小時: 3


In [6]:
pad_top = 8
pad_bottom = 8

# 例如：切分線之間的區域
segments = []
for i in range(len(ys) - 1):
    y1 = max(0, ys[i] + pad_top)
    y2 = min(H, ys[i+1] - pad_bottom)
    if y2 - y1 > 20:
        roi = img[y1:y2, :]
        segments.append(roi)

# 保存看看
for i, roi in enumerate(segments):
    cv2.imshow(f"row_{i:02d}", roi)
    cv2.waitKey(0) 

In [8]:
import os, json, math
import cv2
import numpy as np
from glob import glob

# ------------ 可調參數 ------------
ROOT = r"dataset/mines"         # 你的資料夾：每個子資料夾是一個 label
CROP_RATIO = 0.60               # 中央裁切比例（0.5~0.8 可微調）
PROTO_JSON = "color_prototypes.json"
# ---------------------------------

def center_ab_feature(img_bgr, crop_ratio=0.6):
    """回傳中央區域的平均 a*, b*（OpenCV Lab；a,b 減去 128 作居中）。"""
    h, w = img_bgr.shape[:2]
    cw, ch = int(w*crop_ratio), int(h*crop_ratio)
    x0, y0 = (w - cw) // 2, (h - ch) // 2
    crop = img_bgr[y0:y0+ch, x0:x0+cw]

    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
    a = lab[:,:,1] - 128.0
    b = lab[:,:,2] - 128.0
    return float(a.mean()), float(b.mean())

def build_prototypes(root_dir, crop_ratio=0.6):
    """掃描資料集，為每個 label 建立 a*,b* 的 centroid 與標準差。"""
    prototypes = {}
    for label in sorted(os.listdir(root_dir)):
        label_dir = os.path.join(root_dir, label)
        if not os.path.isdir(label_dir):
            continue

        feats = []
        for fn in os.listdir(label_dir):
            if not fn.lower().endswith((".png",".jpg",".jpeg",".bmp", ".webp")):
                continue
            path = os.path.join(label_dir, fn)
            img = cv2.imread(path)
            if img is None:
                continue
            a, b = center_ab_feature(img, crop_ratio)
            feats.append([a, b])

        if not feats:
            continue

        arr = np.array(feats, dtype=np.float32)
        mean_ab = arr.mean(axis=0)
        std_ab  = arr.std(axis=0) + 1e-6  # 防止 0
        prototypes[label] = {
            "a": float(mean_ab[0]),
            "b": float(mean_ab[1]),
            "std_a": float(std_ab[0]),
            "std_b": float(std_ab[1]),
            "count": int(len(arr))
        }
        print(f"{label}: mean(a*,b*)=({mean_ab[0]:.3f}, {mean_ab[1]:.3f}) | "
              f"std=({std_ab[0]:.3f}, {std_ab[1]:.3f}) | n={len(arr)}")
    return prototypes

def save_prototypes(protos, json_path=PROTO_JSON, notes=None):
    out = {"prototypes": protos, "crop_ratio": CROP_RATIO,
           "notes": notes or "OpenCV Lab a*,b* from central crop; nearest-centroid classification."}
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)
    print(f"Saved prototypes -> {json_path}")

def load_prototypes(json_path=PROTO_JSON):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def classify_ab(a, b, protos):
    """以 (a,b) 與各類原型做最近鄰；回傳 (label, distance)。"""
    best_label, best_dist = None, 1e9
    for label, p in protos.items():
        d = math.hypot(a - p["a"], b - p["b"])
        if d < best_dist:
            best_label, best_dist = label, d
    return best_label, best_dist

def classify_image(img_path, json_path=PROTO_JSON, crop_ratio=CROP_RATIO):
    """對單張圖片做分類。"""
    data = load_prototypes(json_path)
    protos = data["prototypes"]
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Cannot read: {img_path}")
    a, b = center_ab_feature(img, crop_ratio)
    label, dist = classify_ab(a, b, protos)
    return {"path": img_path, "label": label, "distance": dist, "a": a, "b": b}

def classify_folder(folder, json_path=PROTO_JSON, crop_ratio=CROP_RATIO):
    """對一個資料夾內的所有圖片分類。"""
    results = []
    for path in glob(os.path.join(folder, "*")):
        if not path.lower().endswith((".png",".jpg",".jpeg",".bmp", ".webp")):
            continue
        try:
            results.append(classify_image(path, json_path, crop_ratio))
        except Exception as e:
            print(f"[skip] {path} -> {e}")
    return results

if __name__ == "__main__":
    # 1) 由資料集建立/更新原型（每個 label 可能有多張圖）
    protos = build_prototypes(ROOT, CROP_RATIO)
    print("Label count:", len(protos))
    save_prototypes(protos, PROTO_JSON)

    # 2)（可選）示範：拿同資料集的一張圖來測試分類
    # test_img = r"path/to/any/image.jpg"
    # print(classify_image(test_img, PROTO_JSON))


dirt: mean(a*,b*)=(13.778, 33.219) | std=(0.507, 0.841) | n=103
dug_pit: mean(a*,b*)=(3.853, -27.238) | std=(1.276, 1.893) | n=18
empty: mean(a*,b*)=(2.805, 6.786) | std=(0.412, 0.437) | n=69
one_hit_rock: mean(a*,b*)=(0.386, 3.068) | std=(0.252, 0.044) | n=6
reachable_pit: mean(a*,b*)=(-9.656, -23.688) | std=(0.464, 1.937) | n=8
rock: mean(a*,b*)=(-0.001, 3.000) | std=(0.016, 0.001) | n=29
unreachable_dirt: mean(a*,b*)=(8.003, 19.854) | std=(0.126, 0.277) | n=129
unreachable_pit: mean(a*,b*)=(-6.033, -11.965) | std=(1.041, 2.202) | n=12
unreachable_rock: mean(a*,b*)=(-0.063, 2.951) | std=(0.095, 0.075) | n=56
unreachable_void: mean(a*,b*)=(2.517, 3.449) | std=(0.324, 0.278) | n=32
Label count: 10
Saved prototypes -> color_prototypes.json


In [ ]:
# 專門測試檢查次數功能
def test_check_count_functionality():
    """專門測試檢查次數功能"""
    print("=== 檢查次數功能測試 ===")
    manager = create_park_manager("check-count-test")
    
    print("1. 初始狀態 - 應該需要購買")
    should_daily = manager.should_purchase('daily')
    should_weekly = manager.should_purchase('weekly') 
    print(f"   初始 daily 需要購買: {should_daily}")
    print(f"   初始 weekly 需要購買: {should_weekly}")
    
    print("\n2. 第一次記錄檢查 - 檢查次數1")
    manager.record_purchase('daily', buy_num=0, check_time=1)
    manager.record_purchase('weekly', buy_num=0, check_time=1)
    
    print("3. 第二次記錄檢查 - 檢查次數2") 
    manager.record_purchase('daily', buy_num=0, check_time=2)
    manager.record_purchase('weekly', buy_num=0, check_time=2)
    
    print("4. 第三次記錄檢查 - 檢查次數3 (應該被跳過)")
    manager.record_purchase('daily', buy_num=0, check_time=3)
    manager.record_purchase('weekly', buy_num=0, check_time=3)
    
    print("\n5. 檢查最終數據:")
    data = manager.load_data()
    daily_data = data.get('daily', {})
    weekly_data = data.get('weekly', {})
    
    print(f"   Daily 檢查次數: {daily_data.get('car_market_check_time', '未設置')}")
    print(f"   Weekly 檢查次數: {weekly_data.get('car_market_check_time', '未設置')}")
    
    # 使用修正後的 park.py 中的 check 方法來測試
    # 注意：這需要您已經在 park.py 中實現了檢查次數限制邏輯
    
# 執行檢查次數測試
test_check_count_functionality()

In [10]:
import uiautomator2 as u2
import numpy as np
import time
import random
import cv2
import os
from tools import click_white
d = u2.connect('7fe98fc6')  # 你的裝置 ID   
#監聽通知 
img = d.screenshot(format='opencv')[9:39,208:383]
cv2.imwrite("a.jpg", img)


True

In [23]:
import uiautomator2 as u2
import time
import random
import cv2
import img_tools  
d = u2.connect('adb-fc65396d-4LPqmI._adb-tls-connect._tcp')  # 替換為你的設備ID
ip ="fc65396d"
for i in range(10):         
    if 'fc65396d' in ip:
        d.click(random.randint(160, 190),random.randint(145, 179)) #首頁加速的按鈕
    else:
        d.click(178+random.randint(-5, 5),114+random.randint(-5, 5)) #首頁加速的按鈕
    time.sleep(1)
    img = d.screenshot(format='opencv')[239:277,197:340]
    result = img_tools.analyze_skill_via_http(img) #{'combo': '', 'is_unwanted': False, 'normalized_combo': '', 'ocr_results': ['穿越深淵之門'], 'success': True}
    if result ==None or result.get("success") == False:
        print("未識別到文字，請檢查截圖或服務狀態")
        continue
    if result!=None and  result.get('ocr_results')[0] == "戰鬥加速":
        print("找到戰鬥加速")
        break
    img = d.screenshot(format='opencv')[421:470,173:374]
    result = img_tools.analyze_skill_via_http(img)
    print(result) #{'combo': '', 'is_unwanted': False, 'normalized_combo': '', 'ocr_results': ['戰鬥加速'], 'success': True}
    if result == None or result.get("success") == False:
        print("未識別到文字，請檢查截圖或服務狀態")
        continue
    if result != None and result.get('ocr_results')[0] == "游荡哥布林":
        print("游荡哥布林")
        d.click(270,685) #點擊領取
        time.sleep(1)
        d.click(525,10) #關閉


找到戰鬥加速


In [34]:
from json_manager import time_recording, return_time, create_store_manager
ip="adb-fc65396d-4LPqmI._adb-tls-connect._tcp"      
record_time = return_time(ip, name="battle_daily_reset")
print(record_time)

None


In [ ]:
import cv2
def find_car(img):
    img1 = img[652:705]
    
    img = cv2.cvtColor(img1, cv2.COLOR_BGR2HSV)
    # mask = cv2.inRange(img, (37, 0, 0), (179, 255, 255))
    mask = cv2.inRange(img, (0, 101, 114), (179, 255, 255))
    # 膨脹
    mask = cv2.dilate(mask, None, iterations=3)
    # 侵蝕
    mask = cv2.erode(mask, None, iterations=3)
    # 計算輪廓
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # 顯示偵測到的遮罩與原始區域以便除錯（非阻塞）
    try:
        cv2.imshow("find_car_bgr", img1)
        cv2.imshow("find_car_mask", mask)
        cv2.waitKey(0)
    except Exception:
        # 在某些 headless 或非 GUI 環境下，imshow 會失敗；忽略該錯誤以維持流程
        pass
    contours = [cv2.boundingRect(
        contour) for contour in contours if cv2.contourArea(contour) > 1000]
    return contours
d = u2.connect("emulator-5554")
img = d.screenshot(format='opencv')
print(len(find_car(img)))

6


: 

In [ ]:
import cnn_model
import torch
from networkx import center
def load_cnn_model(model_path, num_classes=10):
    # 載入模型
    model = cnn_model.SimpleCNN(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.eval()  # 設定為評估模式
    return model
def img_tools.click_str_by_server(str1: str, d, easyocr_reader):
    img = d.screenshot(format='opencv')
    if not os.path.exists("other_str"):
        os.makedirs("other_str")
    cv2.imwrite("other_str/other_str_{}.jpg".format(time.time()), img)
    result = easyocr_reader.readtext(img)
    for i in result:
        if str1 in str(i[1]):
            [x1, x2, x3, x4] = i[0]
            center = [int((x1[0]+x3[0])/2), int((x1[1]+x3[1])/2)]
            d.click(center[0], center[1])
            return True
    return False
Cnn_model = load_cnn_model("cnn_model.pth")


In [ ]:
import 
def daily_acceleration(d, ip):
    record = return_time(ip, name="daily_acceleration")
    should_execute = False
    if record is None:
        should_execute = True
    else:
        should_execute = record.get("is_next_day", False)
    if should_execute:
        logger.info("執行每日加速")
        d.click(321, 913)
        time.sleep(1)
        while(1):
            cnn_result = cnn_model.predict_image(
                Cnn_model, d.screenshot(format='pillow'))
            if cnn_result == 'homeplace':
                break
        d.click(452,218) #點擊研究中心
        time.sleep(1)
        for i in range(5):
            d.click(168,814) #跳過30分鐘
            time.sleep(0.8)
        d.click(487,923) #點擊返回
        time.sleep(1)
        d.click(321, 919) #點擊家園返回
        time_recording(ip, name="daily_acceleration")
    else:
        logger.info("今日已執行過每日加速，跳過")
d = u2.connect('adb-fc65396d-4LPqmI._adb-tls-connect._tcp')
daily_acceleration(d, ip)

In [ ]:

def img_tools.click_str_by_server(str1: str, d, easyocr_reader):
    img = d.screenshot(format='opencv')
    if not os.path.exists("other_str"):
        os.makedirs("other_str")
    cv2.imwrite("other_str/other_str_{}.jpg".format(time.time()), img)
    result = easyocr_reader.readtext(img)
    for i in result:
        if str1 in str(i[1]):
            [x1, x2, x3, x4] = i[0]
            center = [int((x1[0]+x3[0])/2), int((x1[1]+x3[1])/2)]
            d.click(center[0], center[1])
            return True
    return False
Cnn_model = load_cnn_model("cnn_model.pth")
def oralce(d: u2.Device, easyocr_reader, ip):
    d.click(321, 919)
    retry = 0
    while (retry < 5):
        img = d.screenshot(format='opencv')
        # 顏色檢測 - 家園
        # color check - homeland
        print(cnn_model.predict_image(Cnn_model, d.screenshot(format='pillow')))
        if cnn_model.predict_image(Cnn_model, d.screenshot(format='pillow')) == "homeplace":
            break
        retry += 1
        click_white(d)
    if retry == 5:
        return
    time.sleep(1)
    d.click(101, 158)
    time.sleep(3)
    d.click(500, 174)
    time.sleep(3)
    for _ in range(5):
        d.click(394+random.randint(-3,3),599+random.randint(-3,3)) #+30隻鎬子
    d.click(272, 752)
    time.sleep(3)
    img_tools.click_str_by_server("確定", d, easyocr_reader)
    time.sleep(3)
    click_white(d)
    click_white(d)
    d.click(500, 913)
    time.sleep(3)
    d.click(321, 919)
    time.sleep(3)
oralce(d, easyocr_reader, '7fe98fc6')  # Replace with your device's IP or identifier

homeplace


In [ ]:

from sympy import im


Android_devices = android_devices(d)
import json
def record_json(ip, title, content):
    filename = f"{ip}.json"

    # 讀取檔案，如果不存在就建立空資料
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as file:
            try:
                data = json.load(file)
            except json.JSONDecodeError:
                data = {}
    else:
        data = {}

    # 更新或新增 title 對應的內容
    data[title] = content

    # 寫回檔案
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

    print(f"[OK] 已更新 {filename} 中的欄位 '{title}'。")

def check_json(ip,title):
    filename = f"{ip}.json"
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as file:
            try:
                data = json.load(file)
                if title in data:
                    return data[title]
                else:
                    return None
            except json.JSONDecodeError:
                return None
    else:
        return None

def buy_store(d,ip):
    #檢查是否有紀錄
    record = check_json(ip, "神秘商人")
    #將日期轉換為時間戳
    if record is not None:
        last_time = record.get("last_time")
        if last_time:
            last_time = time.strptime(last_time, "%Y-%m-%d %H:%M:%S")
            current_time = time.localtime()
            #如果距離上次購買小於24小時，則不購買
            if (current_time.tm_year == last_time.tm_year and
                current_time.tm_mon == last_time.tm_mon and
                current_time.tm_mday == last_time.tm_mday):
                print("今天已經購買過了，跳過購買")
                return
    #如果沒有紀錄，或是紀錄已經超過24小時，則購買
    print("開始購買神秘商人")
    d.click(321, 919) #點擊家園
    while(cnn_model.predict_image(Cnn_model, d.screenshot(format='pillow')) != "homeplace"):
        click_white(d)
        time.sleep(1)
    d.click(269, 662) #點擊商店
    time.sleep(2)
    buy_time = 0
    error = 0
    while buy_time<5 and error < 2:
        img = Android_devices.capture_screenshot()
        results = easyocr_reader.readtext(img)
        for result in results:
            if ("每日"in result[1] or "每週" in result[1]) :
                #找出每日限購的中心
                the_center= ((result[0][1][0]+result[0][3][0])//2, (result[0][1][1]+result[0][3][1])//2)
                print(the_center)
                d.click(int(the_center[0]),int( the_center[1])+30)
                time.sleep(1.5+random.random())
                for i in range(4):
                    d.click(389,458)
                d.click(276,555)
                time.sleep(1.5+random.random())
                d.click(523,10)
                buy_time += 1
                if buy_time == 5:
                    break
        d.swipe(261,684,261,100,0.1)
        error +=1
        print("error", error)
    d.click(321, 919) #點擊家園返回
    shop_name ="神秘商人"
    now = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    record_json(ip, shop_name, {
        "count": buy_time,
        "last_time": now
    })
    time.sleep(3)
buy_store(d, '7fe98fc6')  # Replace with your device's IP or identifier

(136, 372)
(347, 372)
(136, 596)
(347, 596)
error 1
(167, 284)
(347, 284)
(136, 508)


KeyboardInterrupt: 

: 

In [1]:
from paddleocr import PaddleOCR
ocr = PaddleOCR(
    lang='chinese_cht',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
    ,ocr_version='PP-OCRv5')



c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in C:\Users\Eric\.paddlex\official_models.
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 2770.04it/s]
Creating model: ('PP-OCRv5_server_rec', None)
Using official model (PP-OCRv5_server_rec), the model files will be automatically downloaded and saved

In [4]:
# 對範例圖片執行 OCR 推論
import cv2
img = cv2.imread(r"A:\ocr_fails\now_stage_low_confidence_0.060_1753959214881.jpg")
result = ocr.predict(
    input=img)
for res in result:
    res.print()

{'res': {'input_path': None, 'page_index': None, 'model_settings': {'use_doc_preprocessor': True, 'use_textline_orientation': False}, 'doc_preprocessor_res': {'input_path': None, 'page_index': None, 'model_settings': {'use_doc_orientation_classify': False, 'use_doc_unwarping': False}, 'angle': -1}, 'dt_polys': array([[[ 2,  3],
        ...,
        [ 2, 19]]], dtype=int16), 'text_det_params': {'limit_side_len': 64, 'limit_type': 'min', 'thresh': 0.3, 'max_side_limit': 4000, 'box_thresh': 0.6, 'unclip_ratio': 1.5}, 'text_type': 'general', 'textline_orientation_angles': array([-1]), 'text_rec_score_thresh': 0.0, 'rec_texts': ['普攻機率'], 'rec_scores': array([0.90676725]), 'rec_polys': array([[[ 2,  3],
        ...,
        [ 2, 19]]], dtype=int16), 'rec_boxes': array([[ 2, ..., 19]], dtype=int16)}}


In [8]:
import cv2
import numpy as np
img = cv2.imread(r"A:\ocr_fails\now_stage_low_confidence_0.060_1753959214881.jpg")
result = ocr.predict(
    input=img)

out = []

# result 是 list，不是 dict，應用索引取出第一個元素
res_obj = result[0]  # 取出內層
texts   = res_obj.get("rec_texts", [])
scores  = res_obj.get("rec_scores", [])
polys   = res_obj.get("rec_polys", None)
boxes   = res_obj.get("rec_boxes", None)

# 轉成 numpy，方便統一處理
texts  = list(texts)
scores = np.asarray(scores) if scores is not None else np.array([])
polys  = np.asarray(polys)  if polys  is not None else None
boxes  = np.asarray(boxes)  if boxes  is not None else None

N = len(texts)

def poly_to_bbox(poly):
    # poly: (K,2) -> [x_min, y_min, x_max, y_max]
    xs = poly[:,0]
    ys = poly[:,1]
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

for i in range(N):
    item = {
        "text": texts[i],
        "score": float(scores[i]) if i < len(scores) else None,
        "polygon": None,
        "bbox": None
    }

    # polygon（若有）
    if polys is not None and len(polys) > i:
        # 常見形狀是 (N, 4, 2)；保險起見再轉一次形狀
        poly_i = np.asarray(polys[i]).reshape(-1, 2).astype(int)
        item["polygon"] = poly_i.tolist()

        # 若沒有 bbox，就由 polygon 算一個
        item["bbox"] = poly_to_bbox(poly_i)

    # bbox（若模型有直接給）
    if boxes is not None and len(boxes) > i:
        # 嘗試辨識是 [x_min, y_min, x_max, y_max] 還是 [x, y, w, h]
        b = np.asarray(boxes[i]).flatten()
        if b.size == 4:
            x1, y1, x2_or_w, y2_or_h = b.astype(int)
            # 用幅度推斷：若 x2_or_w > x1 且 y2_or_h > y1，多半是 x2,y2
            if x2_or_w > x1 and y2_or_h > y1:
                bbox = [x1, y1, x2_or_w, y2_or_h]
            else:
                # 視為 (x, y, w, h)
                bbox = [x1, y1, x1 + x2_or_w, y1 + y2_or_h]
            item["bbox"] = bbox

    out.append(item)

for item in out:
    if item["polygon"] is not None:
        pts = np.array(item["polygon"], dtype=np.int32)
        cv2.polylines(img, [pts], isClosed=True, color=(0,255,0), thickness=2)
    if item["bbox"] is not None:
        x1,y1,x2,y2 = item["bbox"]
        cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)
    cv2.putText(img, f'{item["text"]} {item["score"]:.2f}',
                (item["bbox"][0], max(0, item["bbox"][1]-5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

cv2.imshow("debug_draw.png", img)
cv2.waitKey(0)

-1

In [2]:
# 測試簡化後的技能識別邏輯
import sys
sys.path.append('a:\\菇勇者全自動掛機')

from Open_gold_paddle_ocr import normalize_text, text_to_skill, get_skill_combo, normalize_combo, is_unwanted_combo

# 測試案例
test_cases = [
    ['暴擊', '普攻造成额', '擊量', '普攻使'],  # 您的例子
    ['量', '普攻使方', '連', '普攻機率客'],
    ['閃避', '閃避', '暴擊', '普攻造成客'],
    ['技能暴擊', '反擊', '普攻'],
    ['連擊', '擊暱', '額外'],
    ['回復', '每秒回复百', '技能暴', '技能造成客']
]

for case in test_cases:
    print(f"輸入: {case}")
    
    # 詳細調試每個文字的轉換過程 - 顯示所有文字
    for text in case:
        normalized = normalize_text(text)
        skill = text_to_skill(text)
        print(f"  '{text}' -> '{normalized}' -> '{skill if skill else 'None'}'")
    
    # 先看單個文字的轉換
    skills = [text_to_skill(text) for text in case if text_to_skill(text)]
    print(f"識別技能: {skills}")
    
    # 模擬 OCR 結果格式 [poly, text, score]
    mock_ocr = [[None, text, 0.9] for text in case]
    
    combo = get_skill_combo(mock_ocr)
    normalized = normalize_combo(combo)
    unwanted = is_unwanted_combo(combo)
    
    print(f"組合: {combo} => {normalized}")
    print(f"是否不要: {unwanted}")
    print("---")

輸入: ['暴擊', '普攻造成额', '擊量', '普攻使']
  '暴擊' -> '暴擊' -> '爆'
  '普攻造成额' -> '普攻造成額' -> 'None'
  '擊量' -> '擊暈' -> '暈'
  '普攻使' -> '普攻使' -> 'None'
識別技能: ['爆', '暈']
組合: 爆暈 => 爆暈
是否不要: True
---
輸入: ['量', '普攻使方', '連', '普攻機率客']
  '量' -> '暈' -> '暈'
  '普攻使方' -> '普攻使方' -> 'None'
  '連' -> '連' -> '連'
  '普攻機率客' -> '普攻機率客' -> 'None'
識別技能: ['暈', '連']
組合: 暈連 => 連暈
是否不要: True
---
輸入: ['閃避', '閃避', '暴擊', '普攻造成客']
  '閃避' -> '閃避' -> '閃'
  '閃避' -> '閃避' -> '閃'
  '暴擊' -> '暴擊' -> '爆'
  '普攻造成客' -> '普攻造成客' -> 'None'
識別技能: ['閃', '閃', '爆']
組合: 閃爆 => 連閃
是否不要: False
---
輸入: ['技能暴擊', '反擊', '普攻']
  '技能暴擊' -> '技能暴擊' -> '技'
  '反擊' -> '反擊' -> '反'
  '普攻' -> '普攻' -> 'None'
識別技能: ['技', '反']
組合: 技反 => 技反
是否不要: True
---
輸入: ['連擊', '擊暱', '額外']
  '連擊' -> '連擊' -> '連'
  '擊暱' -> '擊暱' -> 'None'
  '額外' -> '額外' -> 'None'
識別技能: ['連']
組合: 連 => 連
是否不要: False
---
輸入: ['回復', '每秒回复百', '技能暴', '技能造成客']
  '回復' -> '回復' -> '回'
  '每秒回复百' -> '每秒回复百' -> '回'
  '技能暴' -> '技能暴' -> '技'
  '技能造成客' -> '技能造成客' -> '技'
識別技能: ['回', '回', '技', '技']
組合: 回技 => 技回
是否不要: Fa

In [1]:
# 測試 PaddleOCR 初始化
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'  # 強制使用 GPU 0

try:
    from paddleocr import PaddleOCR
    
    # 測試初始化
    ocr = PaddleOCR(
        lang='chinese_cht',
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        ocr_version='PP-OCRv5'
    )
    print("PaddleOCR 初始化成功")
    
    # 測試簡單圖片識別
    import cv2
    import numpy as np
    
    # 創建測試圖片
    test_img = np.ones((100, 200, 3), dtype=np.uint8) * 255
    cv2.putText(test_img, 'TEST', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
    
    result = ocr.predict(input=test_img)
    print("測試識別成功")
    
except Exception as e:
    print(f"PaddleOCR 初始化或測試失敗: {e}")
    import traceback
    traceback.print_exc()

c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


PaddleOCR 初始化或測試失敗: [WinError 127] 找不到指定的程序。 Error loading "c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddle\..\nvidia\cudnn\bin\cublas64_12.dll" or one of its dependencies.


Traceback (most recent call last):
  File "C:\Users\Eric\AppData\Local\Temp\ipykernel_692068\3816089776.py", line 9, in <module>
    ocr = PaddleOCR(
          ^^^^^^^^^^
  File "c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddleocr\_pipelines\ocr.py", line 161, in __init__
    super().__init__(**base_params)
  File "c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddleocr\_pipelines\base.py", line 66, in __init__
    self.paddlex_pipeline = self._create_paddlex_pipeline()
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddleocr\_pipelines\base.py", line 99, in _create_paddlex_pipeline
    kwargs = prepare_common_init_args(None, self._common_args)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Eric\.conda\envs\mushroom\Lib\site-packages\paddleocr\_common_args.py", line 61, in prepare_common_init_args
    device = get_default_device()
             ^^^^^^^^^^^^^^^^^^^

In [43]:
from device import device
import uiautomator2 as u2
import easyocr
import numpy as np
from tools import *
from mask import *
import new_cnn.cnn_model as cnn_model

class New_parking_Manager(device):

    def __init__(self, device, reader):
        super().__init__(device)
        self.device = device
        self.reader = reader



cnn_model_ = cnn_model.load_cnn_model("cnn_model.pth")

easyocr_reader = easyocr.Reader(['ch_tra', 'en'])
ip = "emulator-5554"
d = u2.connect(ip)  # 連接到設備


In [ ]:
import img_tools
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
def stage_by_str(d, ocr_str: list, img: np.ndarray) -> str:
    """

    Args:
        d: The uiautomator2 device object (remains for potential future use).
        ocr_str: A list of strings detected by OCR.
        img: The OpenCV image corresponding to the OCR text.

    Returns:
        The name of the detected stage or "未知".
    """
    # Combine list of strings into a single string for easier searching
    full_text = "".join(ocr_str)

    # Mapping of keywords to stage names and their properties
    stage_keywords = {
        "你的帳號在另一個地方登錄": "異地登錄",
        "退出遊戲": "異地登錄",
        "公告": "公告",
        "方案": "主頁面",
        "放置獎勵": "放置獎勵",
        "離線獎勵": "放置獎勵",
        "家族商店": "家族",
        "家族亂鬥": "家族",
        "征戰熔岩巨獸": "征戰熔岩巨獸",

    }

    for keyword, stage_name in stage_keywords.items():
        # Special condition for "征戰熔岩巨獸"
        
        if stage_name == "征戰熔岩巨獸":
            if "征戰熔岩巨獸" in full_text and "掃蕩" in full_text:
                img_tools.save_stage_debug_image(stage_name, img)
                return stage_name
        # General condition for other stages
        elif keyword in full_text:
            img_tools.save_stage_debug_image(stage_name, img)
            return stage_name
            
    return "未知"
def new_stage_check(img):
    if [abs(np.sum(img[955, 535]) - np.sum([47, 138, 123])) <= 10, abs(np.sum(img[902, 39]) - np.sum([146, 232, 232])) <= 10, abs(np.sum(img[956, 6]) - np.sum([50, 140, 117])) <= 10, abs(np.sum(img[921, 135]) - np.sum([41, 21, 218])) <= 10, abs(np.sum(img[908, 223]) - np.sum([160, 165, 164])) <= 10, abs(np.sum(img[731, 27]) - np.sum([139, 170, 201])) <= 10, abs(np.sum(img[759, 30]) - np.sum([111, 143, 179])) <= 10, abs(np.sum(img[794, 37]) - np.sum([38, 60, 88])) <= 10, abs(np.sum(img[825, 380]) - np.sum([37, 58, 86])) <= 10]:
        return True
    return False
def get_stage(d, Cnn_model, easyocr_reader):
    """ 截圖並判斷目前所在的頁面 """
    # cnn_result = cnn_model.predict_image(
    #     Cnn_model, d.screenshot(format='pillow'))
    # img = d.screenshot(format='opencv')
    # print(new_stage_check(img))

    logger.warning("使用OCR方法")
    result = easyocr_reader.readtext(img, detail=0)
    stage_withocr = stage_by_str(d, result, img)
    if stage_withocr == "異地登錄":
        logger.error("異地登錄，請檢查帳號密碼安全性")
        return "異地登錄"
    # if cnn_result != stage_withocr:
    #     if not os.path.exists("other_stage"):
    #         os.makedirs("other_stage")
    #     cv2.imwrite("other_stage/other_stage_{}.jpg".format(time.time()), img)
    return stage_withocr


In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
測試json_manager模組的功能
"""

from json_manager import (
    create_park_manager, 
    create_time_manager, 
    create_store_manager,
    time_recording,
    return_time,
    record_json,
    check_json
)

def test_park_manager():
    """測試停車場管理器"""
    print("=== 測試停車場管理器 ===")
    manager = create_park_manager("adb-fc65396d-4LPqmI (2)._adb-tls-connect._tcp")
    
    # 測試應該購買（初次）
    print(f"初次檢查是否應該購買daily: {manager.should_purchase('daily')}")
    print(f"初次檢查是否應該購買weekly: {manager.should_purchase('weekly')}")
    
    # 記錄購買
    manager.record_purchase('daily', 1)
    manager.record_purchase('weekly', 1)
    
    # 再次檢查
    print(f"記錄後檢查是否應該購買daily: {manager.should_purchase('daily')}")  # 應該還能購買（< 2）
    print(f"記錄後檢查是否應該購買weekly: {manager.should_purchase('weekly')}")  # 應該還能購買（< 2）
    
    # 獲取購買數據
    data = manager.get_buy_data()
    print(f"購買數據: {data}")
    
def test_time_manager():
    """測試時間記錄管理器"""
    print("\n=== 測試時間記錄管理器 ===")
    manager = create_time_manager("test_device")
    
    # 記錄時間
    manager.record_time("park")
    manager.record_time("Martial_Soul")
    
    # 檢查記錄
    park_record = manager.get_time_record("park")
    martial_record = manager.get_time_record("Martial_Soul")
    
    print(f"停車記錄: {park_record}")
    print(f"武魂記錄: {martial_record}")
    
    # 測試是否為同一天
    print(f"停車記錄是同一天: {manager.is_same_day('park')}")
    print(f"武魂記錄是同一天: {manager.is_same_day('Martial_Soul')}")

def test_store_manager():
    """測試商店管理器"""
    print("\n=== 測試商店管理器 ===")
    manager = create_store_manager("test_device")
    
    # 記錄購買
    import datetime
    now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    manager.record_purchase("挖礦", {"last_time": now_str})
    manager.record_purchase("神秘商人", {"count": 5, "last_time": now_str})
    
    # 檢查記錄
    mining_record = manager.get_purchase_record("挖礦")
    shop_record = manager.get_purchase_record("神秘商人")
    
    print(f"挖礦記錄: {mining_record}")
    print(f"神秘商人記錄: {shop_record}")
    
    # 檢查是否今天已購買
    print(f"挖礦今天已購買: {manager.is_purchased_today('挖礦')}")
    print(f"神秘商人今天已購買: {manager.is_purchased_today('神秘商人')}")
    
    # 檢查是否過期
    print(f"挖礦記錄過期(12小時): {manager.is_purchase_expired('挖礦', hours=12)}")
    print(f"神秘商人記錄過期(24小時): {manager.is_purchase_expired('神秘商人', hours=24)}")

def test_backward_compatibility():
    """測試向後兼容性"""
    print("\n=== 測試向後兼容性 ===")
    
    # 測試舊的函數接口
    time_recording("test_device", "test_action")
    record = return_time("test_device", "test_action")
    print(f"時間記錄: {record}")
    
    import datetime
    now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    record_json("test_device", "test_purchase", {"last_time": now_str})
    purchase_record = check_json("test_device", "test_purchase")
    print(f"購買記錄: {purchase_record}")

def interactive_test():
    """互動式測試"""
    print("JSON 數據管理器互動式測試")
    print("=" * 50)
    
    while True:
        # 獲取用戶輸入的設備 IP/ID
        device_id = input("\n請輸入設備 IP 或 ID (輸入 'quit' 退出): ").strip()
        
        if device_id.lower() in ['quit', 'exit', 'q']:
            print("退出測試程序")
            break
        
        if not device_id:
            print("請輸入有效的設備 ID")
            continue
        
        print(f"\n開始測試設備: {device_id}")
        print("=" * 30)
        
        try:
            # 測試停車場管理器
            print(f"\n=== 停車場管理器測試 (設備: {device_id}) ===")
            park_mgr = create_park_manager(device_id)
            
            # 顯示當前狀態
            daily_ts, daily_num, weekly_ts, weekly_num = park_mgr.get_buy_data()
            print(f"當前狀態:")
            print(f"  Daily - 購買次數: {daily_num}")
            print(f"  Weekly - 購買次數: {weekly_num}")
            
            # 檢查是否需要購買
            should_daily = park_mgr.should_purchase('daily', max_purchases=2)
            should_weekly = park_mgr.should_purchase('weekly', max_purchases=2)
            print(f"  需要每日購買: {should_daily}")
            print(f"  需要每週購買: {should_weekly}")
            
            # 詢問是否要記錄測試購買
            if input("是否要記錄測試購買？(y/n): ").lower() == 'y':
                park_mgr.record_purchase('daily', daily_num + 1, 1)
                park_mgr.record_purchase('weekly', weekly_num + 1, 1)
                print("已記錄測試購買")
            
            # 測試時間記錄管理器
            print(f"\n=== 時間記錄管理器測試 (設備: {device_id}) ===")
            time_mgr = create_time_manager(device_id)
            
            # 記錄測試時間
            time_mgr.record_time("interactive_test")
            print("已記錄 interactive_test 時間")
            
            # 檢查記錄
            record = time_mgr.get_time_record("interactive_test")
            if record:
                print(f"時間記錄: {record}")
                print(f"是否為同一天: {time_mgr.is_same_day('interactive_test')}")
            
            # 測試商店管理器
            print(f"\n=== 商店管理器測試 (設備: {device_id}) ===")
            store_mgr = create_store_manager(device_id)
            
            # 記錄測試購買
            import datetime
            now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            store_mgr.record_purchase("互動測試商品", {
                "quantity": 1,
                "last_time": now_str,
                "test_mode": True
            })
            
            # 檢查記錄
            purchase_record = store_mgr.get_purchase_record("互動測試商品")
            if purchase_record:
                print(f"購買記錄: {purchase_record}")
                print(f"今天已購買: {store_mgr.is_purchased_today('互動測試商品')}")
            
            print(f"\n設備 {device_id} 測試完成!")
            
        except Exception as e:
            print(f"測試過程中發生錯誤: {e}")
            import traceback
            traceback.print_exc()

def show_menu():
    """顯示選單"""
    print("\n請選擇測試模式:")
    print("1. 預設測試 (使用固定設備ID)")
    print("2. 互動測試 (輸入您的設備IP)")
    print("3. 停車場管理器測試")
    print("4. 時間記錄管理器測試")
    print("5. 商店管理器測試")
    print("6. 向後兼容性測試")
    print("0. 退出")
    return input("請輸入選項 (0-6): ").strip()

def run_specific_test():
    """運行特定測試"""
    while True:
        choice = show_menu()
        
        if choice == '0':
            print("退出程序")
            break
        elif choice == '1':
            print("運行預設測試...")
            test_park_manager()
            test_time_manager()
            test_store_manager()
            test_backward_compatibility()
        elif choice == '2':
            interactive_test()
        elif choice == '3':
            test_park_manager()
        elif choice == '4':
            test_time_manager()
        elif choice == '5':
            test_store_manager()
        elif choice == '6':
            test_backward_compatibility()
        else:
            print("無效選項，請重新選擇")

if __name__ == "__main__":
    print("JSON 數據管理器測試程序")
    print("版本: 2.0")
    print("=" * 50)
    
    # 顯示使用說明
    print("\n使用說明:")
    print("- 您可以選擇預設測試或輸入自己的設備IP進行測試")
    print("- 設備IP格式例如: emulator-5554, 192.168.1.100, adb-fc65396d等")
    print("- 所有測試都會在當前目錄創建對應的JSON文件")
    
    run_specific_test()

JSON 數據管理器測試程序
版本: 2.0

使用說明:
- 您可以選擇預設測試或輸入自己的設備IP進行測試
- 設備IP格式例如: emulator-5554, 192.168.1.100, adb-fc65396d等
- 所有測試都會在當前目錄創建對應的JSON文件

請選擇測試模式:
1. 預設測試 (使用固定設備ID)
2. 互動測試 (輸入您的設備IP)
3. 停車場管理器測試
4. 時間記錄管理器測試
5. 商店管理器測試
6. 向後兼容性測試
0. 退出
無效選項，請重新選擇

請選擇測試模式:
1. 預設測試 (使用固定設備ID)
2. 互動測試 (輸入您的設備IP)
3. 停車場管理器測試
4. 時間記錄管理器測試
5. 商店管理器測試
6. 向後兼容性測試
0. 退出
運行預設測試...
=== 測試停車場管理器 ===
初次檢查是否應該購買daily: True
初次檢查是否應該購買weekly: False
已記錄 daily 購買數據，數量: 1, 檢查次數: 0
已記錄 weekly 購買數據，數量: 1, 檢查次數: 0
記錄後檢查是否應該購買daily: True
記錄後檢查是否應該購買weekly: True
購買數據: (1759184240.63627, 1, 1759184240.655634, 1)

=== 測試時間記錄管理器 ===
已記錄 park 的時間戳記: 2025-09-30 06:17:20
已記錄 Martial_Soul 的時間戳記: 2025-09-30 06:17:20
停車記錄: {'timestamp': 1759184240.687209, 'recorded_date': '2025-09-30', 'is_next_day': False}
武魂記錄: {'timestamp': 1759184240.707296, 'recorded_date': '2025-09-30', 'is_next_day': False}
停車記錄是同一天: True
武魂記錄是同一天: True

=== 測試商店管理器 ===
已記錄購買項目 '挖礦': {'last_time': '2025-09-30 06:17:20'}
已記錄購買項目 '神秘商人': {'count': 5, 'last_tim

In [ ]:
# img = d.screenshot(format='pillow')
img = d.screenshot(format='opencv') 

# print(cnn_model.predict_image(cnn_model_, img))
# 185 741
cv2.imshow("debug_draw.png", img[185:741, :]) #將好友車位取出來
cv2.waitKey(0)

32

In [9]:
ip = "7fe98fc6"
device = u2.connect(ip) 
device.click(29, 213)
time.sleep(2)
for _ in range(3):
    rand = random.randint(-5, 5)
    device.click(133+rand, 450+rand)
time.sleep(2)
device.click(368, 515)
time.sleep(2)
# device.click(509, 56)
time.sleep(2)
device.click(364,550)
time.sleep(2)
for _ in range(2):
    device.click(533, 1) # 點擊空白處
    time.sleep(1)

In [2]:
import uiautomator2 as u2
import time
import random
import cv2
import numpy as np
import logging
import mask
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [21]:

def _check_red_dot(d: u2.Device, roi: tuple) -> bool:
    """
    Checks for a red dot within a specified region of interest (ROI) on the screen.

    Args:
        d: The uiautomator2 device object.
        roi: A tuple (y1, y2, x1, x2) defining the ROI to check.

    Returns:
        True if a red dot of sufficient size is found, False otherwise.
    """
    img = d.screenshot(format='opencv')
    if img is None:
        logger.warning("Failed to capture screenshot.")
        return False

    # Crop the image to the region of interest
    y1, y2, x1, x2 = roi
    cropped_img = img[y1:y2, x1:x2]

    # Convert to HSV and create a mask for red color
    hsv = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2HSV)
    lower_red = mask.red_mask_lower
    upper_red = mask.red_mask_upper
    red_mask = cv2.inRange(hsv, lower_red, upper_red)
    # Find contours and check if any are large enough
    contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return any(cv2.contourArea(c) > 47 for c in contours)


def get_Martial_Soul(d: u2.Device):
    """
    Checks for and handles the Martial Soul notification and subsequent actions.
    """
    logger.info("檢查武魂紅點...")
    # ROI for the first red dot check
    roi1 = (472, 590, 455, 507)
    if _check_red_dot(d, roi1):
        logger.info("偵測到武魂紅點，開始處理...")
        d.click(495, 582)
        time.sleep(2)

        # ROI for the second red dot check
        roi2 = (760, 791, 483, 525)
        if _check_red_dot(d, roi2):
            logger.info("偵測到第二個紅點，執行升級流程...")
            d.click(451, 792)
            time.sleep(2)
            d.click(289, 272)
            time.sleep(1.5)
            d.click(474, 254)
            time.sleep(1.5)
            d.click(55, 48)
            time.sleep(1.5)
            d.click(486, 908)
            time.sleep(2)
        
        logger.info("關閉武魂頁面...")
        d.click(271, 883) # Close button
        time.sleep(2)
    else:
        logger.info("未發現武魂紅點。")

In [12]:
ip = "emulator-5556"
device = u2.connect(ip) 
get_Martial_Soul(device)

INFO:__main__:檢查武魂紅點...
INFO:__main__:偵測到武魂紅點，開始處理...


True


INFO:__main__:偵測到第二個紅點，執行升級流程...


True


INFO:__main__:關閉武魂頁面...


In [14]:
roi1 = (472, 590, 507, 528)
if _check_red_dot(device, roi1):
    print("有紅點")

True
有紅點


In [ ]:
#守護靈
import tools

def get_Guardian_Spirit(d: u2.Device):
    #守護靈紅點檢查
    roi = (472, 590, 507, 528)
    if _check_red_dot(d, roi):
        print("有紅點")
        d.click(525+random.randint(-5, 5), 595+random.randint(-5, 5))
        time.sleep(2)
        d.click(145+random.randint(-50, 50), 600+random.randint(-50, 50))
        time.sleep(2)
        if _check_red_dot(d, (881, 910, 426, 465)):
            print("偵測到守護靈紅點，開始處理...")
            d.click(404+random.randint(-20, 20),908+random.randint(-8, 8))  #點擊召喚按鈕
            time.sleep(2)
            d.click(478+random.randint(-5, 5),110+random.randint(-5, 5))    #點擊每日10/10
            time.sleep(2)
            d.click(393,467)  #點擊+10按鈕
            time.sleep(2)
            d.click(218+random.randint(-5, 5),542+random.randint(-5, 5))  #點擊購買按鈕
            time.sleep(2)
            #點擊空白處
            tools.click_white(d)
            time.sleep(2)
            #點擊召喚5次的按鈕
            for _ in range(4):
                d.click(330+random.randint(-5, 5),780+random.randint(-5, 5))
                time.sleep(1)
                tools.click_white(d)
                time.sleep(0.5)
            #點擊招喚1次的按鈕
            tools.click_white(d)
            for _ in range(1):
                d.click(149+random.randint(-5, 5),780+random.randint(-5, 5))
                time.sleep(1)
                tools.click_white(d)
                time.sleep(0.5)
            tools.click_white(d)
            tools.click_white(d)
            time.sleep(0.5)
        else:
            print("未偵測到守護靈紅點。")
        d.click(500+random.randint(-5, 5),922+random.randint(-5, 5))    #點擊退出按鈕
        time.sleep(2)
    d.click(18,959)     #點擊回到主頁  
    time.sleep(2)  
get_Guardian_Spirit(device)

有紅點


In [1]:
import uiautomator2 as u2
import easyocr
import numpy as np
import time
import random
import cv2
import os
from tools import click_white
d = u2.connect('fc65396d')

In [ ]:
# //*[@resource-id="com.android.systemui:id/bubble_view"] 要點擊的元件


有FB
906 94 1080 309
{'bottom': 309, 'left': 906, 'right': 1080, 'top': 94}
993.0 201.5


In [45]:
import uiautomator2 as u2
import time
from datetime import datetime
from img_tools import *
d = u2.connect('7fe98fc6')  # Replace with your device's IP or identifier

In [53]:
def find_car( img):
        img1 = img[652+40:705+30]
        # cv2.imshow("debug_draw.png", img1)
        # cv2.waitKey(0)
        img = cv2.cvtColor(img1, cv2.COLOR_BGR2HSV)
        # mask = cv2.inRange(img, (37, 0, 0), (179, 255, 255))
        mask = cv2.inRange(img, (0, 101, 114), (179, 255, 255))
        # 膨脹
        mask = cv2.dilate(mask, None, iterations=2)
        # 侵蝕
        mask = cv2.erode(mask, None, iterations=1)
        # 計算輪廓
        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # 顯示偵測到的遮罩與原始區域以便除錯（非阻塞）
        # try:
        #     cv2.imshow("find_car_bgr", img1)
        #     cv2.imshow("find_car_mask", mask)
        #     cv2.waitKey(0)
        # except Exception:
        #     # 在某些 headless 或非 GUI 環境下，imshow 會失敗；忽略該錯誤以維持流程
        #     pass
        contours = [cv2.boundingRect(
            contour) for contour in contours if cv2.contourArea(contour) > 1000]
        return contours

img = d.screenshot(format='opencv')
# contours = find_car(img)
# print(len(contours))
# #框出來
# for (x, y, w, h) in contours:
#     cv2.rectangle(img, (x, y+652+30), (x + w, y + h+652+30), (0, 255, 0), 2)
cv2.imshow("debug_draw.png", img[372:442, 380:465])
cv2.waitKey(0)

32

In [58]:
import easyocr
easyocr_reader = easyocr.Reader(['ch_tra', 'en'])
print(easyocr_reader.readtext(img[372:442, 380:465], detail=0))
def process_hsv(img, lower, upper, roi=None):
    """HSV 過濾處理，支持 ROI（感興趣區域）"""
    if roi:
        img = img[roi[1]:roi[3], roi[0]:roi[2]]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower, upper)
    return mask
def detect_contours(mask, min_area=1500):
    """檢測輪廓，根據最小面積過濾"""
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # #print([cv2.contourArea(contour) for contour in contours])
    return [contour for contour in contours if cv2.contourArea(contour) > min_area]
import datetime
def count_cars(img):
    """計算停車數量"""
    roi = (0, 138, img.shape[1], 259)  # 限定感興趣區域
    lower = np.array([15,  0, 180])   # H, S, V 下界
    upper = np.array([35, 65, 255])   # H, S, V 上界（S 限制在 65 以下）
    mask = process_hsv(img, lower, upper, roi)
    # 膨脹
    mask = cv2.dilate(mask, None, iterations=2)
    # 侵蝕
    mask = cv2.erode(mask, None, iterations=2)
    contours = detect_contours(mask, min_area=1400)
    return contours
def check_if_12hour(device, reader):
    if datetime.datetime.now().hour < 12:
        return
    img = device.screenshot(format='opencv')
    for car in count_cars(img):
        x, y, w, h = cv2.boundingRect(car)
        # 計算中心點座標
        center = (int(x + w / 2), int(y + h / 2)+138)
        device.click(center[0], center[1])
        time.sleep(2)
        img = device.screenshot(format='opencv')
        result = reader.readtext(img[372:442, 380:465], detail=0)
        print("ocr:", result)
        if 'O/' in str( result) or '0/' in str(result):
            #print("full")
            device.click(375, 744)
            time.sleep(2)
            device.click(379, 552)
            time.sleep(2)
        device.click(509, 56)
        time.sleep(2)
check_if_12hour(d, easyocr_reader)

['0/h']
ocr: ['0/h']
ocr: ['0/h']
ocr: ['0/h']
ocr: ['0/h']


KeyboardInterrupt: 

In [ ]:
import cv2
import uiautomator2 as u2
import time
import BUY
import random
import opencc
import img_tools  

d= u2.connect('emulator-5558')
def into_cloud(d):
    img_tools.click_str_by_server(d,'副本')
    time.sleep(2)
    for _ in range(3):
        d.swipe(239,752,239,352,0.2)
        time.sleep(0.1)
    d.click(239, 752)
    d.swipe(239,262,239,310,0.2)
    time.sleep(1)
    d.click(239, 752)
    state = False
    if img_tools.click_str_by_server(d,'雲纏天梯試煉',(426-149),(410-335)):
        print("點擊成功")
        state = True
        if img_tools.click_str_by_server(d,'結算獎勵'):
            print("點擊成功")
    return state

def friend_help(d,name='大車輪'):
    '''請求朋友幫助'''
    img_tools.click_str_by_server(d,'戰友設置',y_range=(726,827))
    time.sleep(2)
    img_tools.click_str_by_server(d,'戰友招募',y_range=(583,631))
    time.sleep(0.1)
    img_tools.click_str_by_server(d,name)
    time.sleep(2)
    img_tools.click_str_by_server(d,'發送')
    time.sleep(0.1)
    d.click(60,828)
    time.sleep(0.5)
    d.click(276,888)
    img_tools.click_str_by_server(d,'關閉')
def help_friend(d):
    '''幫助朋友'''
    img_tools.click_str_by_server(d,'助戰設置',y_range=(726,827))
    time.sleep(2)
    check = True
    img = d.screenshot(format='opencv')
    result = analyze_skill_via_http(img[607:740,:])
    while img_tools.click_str_by_server(d,'新申請',y_range=(606,740)):
        time.sleep(1)
        img_tools.click_str_by_server(d,'同意')
        time.sleep(1)
        d.swipe(300,673,259,673,0.3)
        time.sleep(0.3)
    d.click(276,888)   
    time.sleep(1)
    img_tools.click_str_by_server(d,'關閉')
def cloud_fight_pre(d,ip,name='大車輪'):
    state = into_cloud(d)
    if not state:
        return
    time.sleep(2)
    if 'emulator-5558' not in ip:
        friend_help(d,name)
        time.sleep(2)
def cloud_fight_pre_help(d):
    time.sleep(2)
    help_friend(d)
    time.sleep(2)
def check_if_pass(d):
    img = d.screenshot(format='opencv')
    result = analyze_skill_via_http(img)
    if result['success']!=False:
        for text in result['ocr_results']:
            if '已通过最高难度' in text['text']:
                print("已通過最高難度，跳過")
                return True
    return False

def cloud_fighting(d,ip,name='大車輪'):
    if 'emulator-5558' in ip:
        switch_skill(d)
    state = into_cloud(d)
    if check_if_pass(d) and state:
        d.click(276,888)   
        time.sleep(1)
        img_tools.click_str_by_server(d,'關閉')
        return
    img_tools.click_str_by_server(d,'戰友設置',y_range=(726,827))
    time.sleep(2)
    img_tools.click_str_by_server(d,'選擇')
    time.sleep(0.2)
    img_tools.click_str_by_server(d,'副本入場')
    time.sleep(2)
    for i in range(5):
        if check_if_pass(d):
            break
        img_tools.click_str_by_server(d,'入場')
        time.sleep(2)
        MAX_CHALLENGE_WAIT = 300  # 最多等 60 秒，你可自行調整
        while img_tools.click_str_by_server(d, '前往挑戰'):
            time.sleep(2)
            img_tools.click_str_by_server(d, '開始挑戰')
            time.sleep(7)
            start = time.time()
            while True:
                # 若已經看到「挑戰成功」，跳出內層迴圈
                if img_tools.click_str_by_server(d, '挑戰成功'):
                    break
                # timeout 檢查
                if time.time() - start > MAX_CHALLENGE_WAIT:
                    print("挑戰等待超過 timeout，放棄這一場")
                    break
                if img_tools.click_str_by_server(d, '恭喜獲得'):
                    break
                time.sleep(2)
    d.click(276,888)   
    time.sleep(1)
    img_tools.click_str_by_server(d,'關閉')
def switch_skill(d,skill_name='戰士推圖'):
    img_tools.click_str_by_server(d,'方案')
    img_tools.click_str_by_server(d,'冒險行裝',shift_y=80)
    img_tools.click_str_by_server(d,skill_name)
    time.sleep(2)
    img_tools.click_str_by_server(d,'切換方案')
    d.click(275,870)
    time.sleep(2)
# cloud_fighting(d,'emulator-5554','大車輪')
# # if result['success']!=False:
# #     for text in result['ocr_results']:
# #         print("識別文字:", text['text'])
#         # if 
# #     skills = result['skills']
# #     print("技能組合:", skills)
# #     unwanted = BUY.is_unwanted_combo(skills)
# #     print("是否不要:", unwanted)
# #     if not unwanted:
# #         img_tools.click_str_by_server(d,'使用該戰友',y_range=(719,740))
# #         time.sleep(0.5)
# #         d.click(60,828)
# #         time.sleep(0.5)
# #         d.click(276,888)
# #     else:
# #         print("不要此戰友，跳過")
# # # while()
# # if img_tools.click_str_by_server(d,'Lv',y_range=(719,740)):
# #     if 

In [1]:
import cv2
import uiautomator2 as u2
import time
import BUY
import random
import opencc
import img_tools
# def click_str_by_server_by_server(d: u2.Device, target_str: str, shift_x=0, shift_y=0) -> bool:
#     img = d.screenshot(format='opencv')
#     result = analyze_skill_via_http(img)
#     ocr_results = result['ocr_results']
#     #將文字都轉為繁體
#     for item in ocr_results:
#         item['text'] = opencc.OpenCC('s2t').convert(item['text'])
#         # print("找到文字:", item['text'])
#         # print("位置:", item['bbox'])
#         if target_str in item['text']:
#             print("找到文字:", item['text'])
#             print("位置:", item['bbox'])
#             x1, y1, x2, y2 = item['bbox']
#             d.click((x1 + x2) // 2 + shift_x, (y1 + y2) // 2 + shift_y)
#             cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
#             return True
#     return False
#競技場購買
def buy_arean_everyday(d):
    img_tools.click_str_by_server(d, '競技場',shift_y=-20)
    time.sleep(2)
    img_tools.click_str_by_server(d,'跨服')
    time.sleep(2)
    img_tools.click_str_by_server(d,'商城')
    want_items = ['覺醒水晶']
    BUY.buy_items(d, want_items)
    d.click(31,918)
    time.sleep(0.5)
    d.click(491,909)
    time.sleep(2)
def _update_store_record_extra(manager, title, extra_fields):
    """輔助更新商店記錄，不改動時間戳與 schema。"""
    data = manager.load_data()
    record = data.get(title, {})
    record.update(extra_fields)
    data[title] = record
    manager.save_data(data)
# def buy_god_everweek(d):
#     # 檢查並購買萬神秘寶閣物品 要跨週才會購買
#     store_manager = create_store_manager(d.adb_device.info.get('serialno'))
#     TPE = datetime.timezone(datetime.timedelta(hours=8))
#     today_str = datetime.datetime.now(TPE).strftime("%Y-%m-%d")
#     record = store_manager.get_purchase_record("萬神_秘寶閣") or {}
#     # 每次呼叫都更新檢查次數（跨日重置）
#     last_check_date = record.get("last_check_date")
#     check_times = int(record.get("check_times", 0))
#     if last_check_date != today_str:
#         _update_store_record_extra(
#         store_manager,
#         "萬神_秘寶閣",
#         {"check_times": check_times, "last_check_date": today_str},
#     )
#     if store_manager.is_purchased_today("萬神_秘寶閣","week"):
#         print("今天已經購買過萬神秘寶閣，跳過")
#         return
#      # 點擊萬神秘寶閣並購買物品
#     img_tools.click_str_by_server(d, '秘寶閣',shift_y=-20)
#     time.sleep(2)
#     want_items = ['神樹精華','符石還原劑','靈契之符',]
#     result  = BUY.buy_items(d,want_items)
#     _update_store_record_extra
#     time.sleep(1)

#     img_tools.click_str_by_server(d,'秘寶閣',shift_y=844-93)

    # record_json(d.adb_device.info.get('serialno'), '每週萬神秘寶閣商店', result)
d = u2.connect('adb-fc65396d-4LPqmI._adb-tls-connect._tcp')
# from new_battle import buy_god_everyweek
# buy_god_everyweek(d)
print(d.adb_device.info.get('serialno'))
# img_tools.click_str_by_server(d, '秘寶閣', shift_y=-20)

# buy_god_everweek(d)
# def fight_test(d):
#     img_tools.click_str_by_server(d,'副本')
#     time.sleep(2)
#     for _ in range(3):
#         d.swipe(239,752,239,352,0.2)
#     d.click(239, 752)

#     time.sleep(2)
#     if img_tools.click_str_by_server(d,'萬神試煉',(426-149),(410-335)):
#         time.sleep(2)
#         for i in range(7):
#             img_tools.click_str_by_server(d,'開始')
#             time.sleep(1)
#             img_tools.click_str_by_server(d,'開始')
#             time.sleep(1)
#             img_tools.click_str_by_server(d,'確定')
#             time.sleep(1.5)
#             d.click(446,81)
#             img_tools.click_str_by_server(d,'開始挑戰')
#             time.sleep(7)
#             img_tools.click_str_by_server(d,'跳過')
#             time.sleep(2)
#             d.click(446,81)
#             time.sleep(2)
#             d.click(490,919)#點擊退出
#             time.sleep(1)
#             img_tools.click_str_by_server(d,'結束本局',170-339,521-433)
#             time.sleep(1)
#             img_tools.click_str_by_server(d,'確定')
#             time.sleep(1.5)
#             d.click(446,81)
#             time.sleep(0.5)
#             d.click(274,875)
#             time.sleep(1)
#         d.click(490,919)#點擊退出
#         time.sleep(1)
#         img_tools.click_str_by_server(d,'關閉')
# fight_test(d)

# img_tools.click_str_by_server(d,'覺醒卷轴',-169)
# time.sleep(2)
# cllick_str(d,'覺醒卷軸',-169)
# img = d.screenshot(format='opencv')
# result = analyze_skill_via_http(img)
# if result ==404:
#     print("OCR服務無法連接")
# else:
#     print("OCR結果:")
#     for item in result['ocr_results']:
#         # print(item['text'])
#         if '競技場' in item['text']:
#             print("找到競技場文字:", item['text'])
#             print("位置:", item['bbox'])
#             x1, y1, x2, y2 = item['bbox']
#             d.click((x1 + x2) // 2, (y1 + y2) // 2-20)
            # cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
    # cv2.imshow("debug_draw.png", img)
    # cv2.waitKey(0)
# for item in result['ocr_results']:
#     print(item['text'])


ConnectError: device adb-fc65396d-4LPqmI._adb-tls-connect._tcp not online

In [2]:
img_tools.click_str_by_server(d,'萬神試煉',(426-149),(410-335))


NameError: name 'd' is not defined

找到文字: 雪國危機 在位置: [463, 11, 528, 30]
OCR 結果: [{'bbox': [463, 11, 528, 30], 'bbox_rel': [0.8574074074074074, 0.09565217391304348, 0.9777777777777777, 0.2608695652173913], 'score': 0.8851760029792786, 'text': '雪國危機'}]
找到文字: 雪國危機
位置: [463, 11, 528, 30]
調整後位置: (463, 727, 528, 746)
OCR 結果: [{'bbox': [405, 7, 457, 38], 'bbox_rel': [0.75, 0.15217391304347827, 0.8462962962962963, 0.8260869565217391], 'score': 0.8303629159927368, 'text': '入場'}]
找到文字: 入場
位置: [405, 7, 457, 38]
調整後位置: (405, 257, 457, 288)
OCR 結果: [{'bbox': [222, 8, 318, 37], 'bbox_rel': [0.4111111111111111, 0.1702127659574468, 0.5888888888888889, 0.7872340425531915], 'score': 0.9105339050292969, 'text': '前往組隊'}]
找到文字: 前往組隊
位置: [222, 8, 318, 37]
調整後位置: (222, 756, 318, 785)
OCR 結果: [{'bbox': [145, 7, 200, 37], 'bbox_rel': [0.26851851851851855, 0.14893617021276595, 0.37037037037037035, 0.7872340425531915], 'score': 0.9901661276817322, 'text': '速戰'}, {'bbox': [357, 8, 410, 37], 'bbox_rel': [0.6611111111111111, 0.1702127659574468, 0.7592

KeyboardInterrupt: 

In [ ]:
d = u2.connect('emulator-5554')
# img_tools.click_str_by_server(d,'討伐結束',y_range=(0,308))   



OCR 結果: [{'bbox': [0, 1, 48, 18], 'bbox_rel': [0.0, 0.003246753246753247, 0.08888888888888889, 0.05844155844155844], 'score': 0.9748479723930359, 'text': 'P:0/1'}, {'bbox': [74, 0, 134, 19], 'bbox_rel': [0.13703703703703704, 0.0, 0.24814814814814815, 0.06168831168831169], 'score': 0.9493547081947327, 'text': 'dX:19.0'}, {'bbox': [152, 1, 213, 17], 'bbox_rel': [0.2814814814814815, 0.003246753246753247, 0.39444444444444443, 0.05519480519480519], 'score': 0.9229570627212524, 'text': 'dY:-75.9'}, {'bbox': [231, 2, 277, 16], 'bbox_rel': [0.42777777777777776, 0.006493506493506494, 0.512962962962963, 0.05194805194805195], 'score': 0.9209217429161072, 'text': 'Xv:0.0'}, {'bbox': [307, 1, 369, 17], 'bbox_rel': [0.5685185185185185, 0.003246753246753247, 0.6833333333333333, 0.05519480519480519], 'score': 0.9622849822044373, 'text': 'Yv:0.071'}, {'bbox': [384, 1, 439, 18], 'bbox_rel': [0.7111111111111111, 0.003246753246753247, 0.812962962962963, 0.05844155844155844], 'score': 0.9443548917770386, '

True

In [ ]:
from opencc import OpenCC
import cv2
import json
import time
import random
cc = OpenCC('t2s')   # 繁 -> 簡


# 模糊比對
def fuzzy_match(ocr_text, target_list, min_match_ratio=0.6):
    if not ocr_text:
        return False

    ocr_s = cc.convert(ocr_text.lower())

    for target in target_list:
        t = cc.convert(target.lower())

        hit = sum(1 for ch in t if ch in ocr_s)

        if hit / len(t) >= min_match_ratio:
            return True

    return False


def buy_items(d: u2.Device, wanted_items: list, stop_str=''):
    
    # 統一 wanted_items → 全部轉簡體
    wanted_items_s = [cc.convert(w) for w in wanted_items]

    def is_same_card_rel(box_rel_a, box_rel_b, x_th=0.05, y_th=0.18):
        ax1, ay1, ax2, ay2 = box_rel_a
        bx1, by1, bx2, by2 = box_rel_b

        acx = (ax1 + ax2) / 2
        acy = (ay1 + ay2) / 2
        bcx = (bx1 + bx2) / 2
        bcy = (by1 + by2) / 2

        dx = abs(acx - bcx)
        dy = abs(acy - bcy)

        return dx < x_th and dy < y_th

    bought_items = []     
    skipped_items = []    
    missing_items = []    

    for retry in range(3):
        print(f"第 {retry+1} 次掃描…")

        img = d.screenshot(format='opencv')
        result = analyze_skill_via_http(img)

        if not result["success"]:
            time.sleep(1)
            continue

        titles = []
        sold_flags = []

        for item in result['ocr_results']:
            txt = item['text']
            txt_s = cc.convert(txt)

            if fuzzy_match(txt, wanted_items):
                titles.append(item)

            if fuzzy_match(txt, ['已售罄', '已售', '售罄']):
                sold_flags.append(item)

        card_status = []

        for t in titles:
            sold = False
            t_rel = t["bbox_rel"]

            for s in sold_flags:
                if is_same_card_rel(t_rel, s["bbox_rel"]):
                    sold = True
                    break

            card_status.append({
                "name": cc.convert(t["text"]),  # ★ 統一簡體
                "bbox": t["bbox"],
                "sold_out": sold
            })

        for card in card_status:
            name_s = card["name"]  # 簡體名字

            if name_s in bought_items or name_s in skipped_items:
                continue

            if card["sold_out"]:
                print(f"{name_s} 已售罄 → 跳過")
                skipped_items.append(name_s)
                continue

            x1, y1, x2, y2 = card["bbox"]
            buy_x = (x1 + x2) // 2
            buy_y = y2 + 163

            print(f"購買 {name_s} 於 ({buy_x}, {buy_y})")
            d.click(buy_x, buy_y)
            time.sleep(0.5)
            d.click(392+random.randint(-2, 2), 460+random.randint(-2, 2))
            time.sleep(0.5)
            d.click(271,557)    
            time.sleep(1)
            click_white(d)
            bought_items.append(name_s)

        # 全部處理完
        processed = set(bought_items) | set(skipped_items)
        if all(cc.convert(w) in processed for w in wanted_items):
            break

        d.swipe(500, 800, 500, 400, 0.5)
        time.sleep(0.5)

    # 最後缺少的
    for w in wanted_items_s:
        if w not in bought_items and w not in skipped_items:
            missing_items.append(w)

    return {
        "bought": bought_items,
        "sold_out": skipped_items,
        "missing": missing_items
    }

wanted_items = ['改裝指南', '神力水晶', '離線躍遷卡']  # 可購買物品名稱列表

buy_items(d, wanted_items)

第 1 次掃描…
改装指南 已售罄 → 跳過
購買 神力水晶 於 (379, 449)
离线跃迁卡 已售罄 → 跳過


{'bought': ['神力水晶'], 'sold_out': ['改装指南', '离线跃迁卡'], 'missing': []}

In [ ]:
import uiautomator2 as u2
import cv2
import time
import random
from img_tools import analyze_skill_via_http, click_str_by_server,check_str_in_region,wait_for_any_text
d= u2.connect('emulator-5554')
img = d.screenshot(format='opencv')
def buy_gift_for_friend(d,ip_str='emulator-5554'):
    if ip_str =='emulator-5556':
        return 
    click_str_by_server(d,'家園',shift_y=-20,y_range=(934,959))
    click_str_by_server(d,'比格先生',shift_y=-20,wait_timeout=5,y_range=(0,195))
    click_str_by_server(d,'贈禮',shift_y=-20,wait_timeout=5,y_range=(258,300))
    click_str_by_server(d,'+10',wait_timeout=5,y_range=(499,544),shift_x=random.randint(-5, 5),shift_y=random.randint(-5, 5))
    click_str_by_server(d,'使用',wait_timeout=5,y_range=(635,690),shift_x=random.randint(-5, 5),shift_y=random.randint(-5, 5))
    click_str_by_server(d,'使用',wait_timeout=5,y_range=(635,690),shift_x=random.randint(-5, 5),shift_y=70)
    # click_str_by_server(d,'贈禮',shift_x=-433+155,wait_timeout=5,y_range=(258,300))
    time.sleep(0.3)
    d.click(156,260)
    click_str_by_server(d,'切磋',wait_timeout=5,y_range=(721,771))
    result = wait_for_any_text(
        d, 
        text_list=['跳過', '勝利', '失敗'],
        timeout=30,  # 最多等待30秒
        check_interval=0.5,  # 每0.5秒檢查一次
        y_range=(167, 771),  # 涵蓋所有可能的Y範圍
        click_if_found=True  # 找到後自動點擊
    )
    click_str_by_server(d,'舉報',wait_timeout=5,shift_x=60,shift_y=70,y_range=(784,846))
    time.sleep(2)
for _ in range(10):
    buy_gift_for_friend(d)


# click_str_by_server(d,'跳過',wait_timeout=20,y_range=(721,771))





# 每日任務管理系統

使用 `json_manager` 來管理每日任務的執行狀態，確保任務每天只執行一次。

In [ ]:
# 初始化設備和 JSON 管理器
import uiautomator2 as u2
from json_manager import JsonDataManager
from img_tools import wait_for_any_text
import time

# 連接設備
device_id = 'emulator-5554'  # 請修改為您的設備ID
d = u2.connect(device_id)

# 創建 JSON 管理器（用於記錄每日任務）
manager = JsonDataManager(device_id, file_suffix="daily_tasks")

print(f"設備已連接: {device_id}")
print(f"JSON 文件: {manager.get_filename()}")

In [ ]:
# 定義每日任務函數
def daily_battle_task(d, manager, task_name="battle_task"):
    """
    每日戰鬥任務
    
    Args:
        d: uiautomator2 設備對象
        manager: JsonDataManager 管理器
        task_name: 任務名稱（用於記錄）
    
    Returns:
        bool: 任務是否執行成功
    """
    # 檢查今天是否已執行過
    if manager.is_same_day(task_name):
        print(f"✓ {task_name} 今天已執行過，跳過")
        record = manager.get_record(task_name)
        print(f"  上次執行時間: {record.get('datetime', 'Unknown')}")
        return False
    
    print(f"開始執行 {task_name}...")
    
    try:
        # === 這裡放您的戰鬥邏輯 ===
        
        # 等待戰鬥結束（跳過/勝利/失敗）
        result = wait_for_any_text(
            d, 
            text_list=['跳過', '勝利', '失敗'],
            timeout=60,
            check_interval=0.5,
            y_range=(167, 771),
            click_if_found=True
        )
        
        if result:
            print(f"戰鬥結束: {result}")
            
            # 記錄執行成功
            manager.record_timestamp(task_name, {
                "status": "success",
                "result": result
            })
            
            print(f"✓ {task_name} 執行完成並已記錄")
            return True
        else:
            print(f"✗ {task_name} 執行超時")
            
            # 即使超時也記錄（避免重複執行）
            manager.record_timestamp(task_name, {
                "status": "timeout"
            })
            return False
            
    except Exception as e:
        print(f"✗ {task_name} 執行時發生錯誤: {e}")
        
        # 發生錯誤時也記錄
        manager.record_timestamp(task_name, {
            "status": "error",
            "error": str(e)
        })
        return False

# 執行任務
daily_battle_task(d, manager, "battle_task")

## 每日贈禮任務

為好友贈禮並切磋，使用 `json_manager` 確保每天只執行一次。

In [ ]:
import uiautomator2 as u2
from json_manager import JsonDataManager
from img_tools import click_str_by_server, wait_for_any_text
import time
import random

def buy_gift_for_friend_once(d, ip_str='emulator-5554'):
    """
    單次執行贈禮給好友的邏輯
    
    Args:
        d: uiautomator2 設備對象
        ip_str: 設備IP字串
    
    Returns:
        bool: 是否執行成功
    """
    if ip_str == 'emulator-5556':
        print("設備 emulator-5556 跳過此任務")
        return False
    
    try:
        click_str_by_server(d, '家園', shift_y=-20, y_range=(934, 959))
        click_str_by_server(d, '比格先生', shift_y=-20, wait_timeout=5, y_range=(0, 195))
        click_str_by_server(d, '贈禮', shift_y=-20, wait_timeout=5, y_range=(258, 300))
        click_str_by_server(d, '+10', wait_timeout=5, y_range=(499, 544), 
                          shift_x=random.randint(-5, 5), shift_y=random.randint(-5, 5))
        click_str_by_server(d, '使用', wait_timeout=5, y_range=(635, 690), 
                          shift_x=random.randint(-5, 5), shift_y=random.randint(-5, 5))
        click_str_by_server(d, '使用', wait_timeout=5, y_range=(635, 690), 
                          shift_x=random.randint(-5, 5), shift_y=70)
        
        time.sleep(0.3)
        d.click(156, 260)
        
        click_str_by_server(d, '切磋', wait_timeout=5, y_range=(721, 771))
        
        # 等待戰鬥結束
        result = wait_for_any_text(
            d, 
            text_list=['跳過', '勝利', '失敗'],
            timeout=30,
            check_interval=0.5,
            y_range=(167, 771),
            click_if_found=True
        )
        
        if result:
            print(f"戰鬥結果: {result}")
        
        click_str_by_server(d, '舉報', wait_timeout=5, shift_x=60, shift_y=70, y_range=(784, 846))
        time.sleep(2)
        
        return True
        
    except Exception as e:
        print(f"執行贈禮任務時發生錯誤: {e}")
        return False


def buy_gift_for_friend_daily(d, ip_str, times=1):
    """
    每日贈禮任務（自動管理執行狀態）    
    
    Args:
        d: uiautomator2 設備對象
        ip_str: 設備IP字串
        times: 每日執行次數（默認1次）
    
    Returns:
        dict: 執行結果統計
    """
    # 初始化 JSON 管理器
    manager = JsonDataManager(ip_str, file_suffix="gift_task")
    task_name = "daily_gift_friend"
    
    # 檢查今天是否已執行過
    if manager.is_same_day(task_name):
        record = manager.get_record(task_name)
        print(f"✓ 今日贈禮任務已完成")
        print(f"  執行時間: {record.get('datetime', 'Unknown')}")
        print(f"  成功次數: {record.get('success_count', 0)}/{record.get('total_count', 0)}")
        return {
            "executed": False,
            "reason": "already_done_today",
            "last_record": record
        }
    
    print(f"開始執行每日贈禮任務（共 {times} 次）...")
    
    success_count = 0
    fail_count = 0
    
    for i in range(times):
        print(f"\n--- 第 {i+1}/{times} 次贈禮 ---")
        
        if buy_gift_for_friend_once(d, ip_str):
            success_count += 1
            print(f"✓ 第 {i+1} 次成功")
        else:
            fail_count += 1
            print(f"✗ 第 {i+1} 次失敗")
        
        # 間隔一下避免操作過快
        if i < times - 1:
            time.sleep(1)
    
    # 記錄執行結果
    manager.record_timestamp(task_name, {
        "status": "completed",
        "total_count": times,
        "success_count": success_count,
        "fail_count": fail_count
    })
    
    print(f"\n{'='*50}")
    print(f"每日贈禮任務完成！")
    print(f"成功: {success_count} 次 | 失敗: {fail_count} 次")
    print(f"{'='*50}")
    
    return {
        "executed": True,
        "total_count": times,
        "success_count": success_count,
        "fail_count": fail_count
    }


# 使用示例
device_id = 'emulator-5554'
d = u2.connect(device_id)

# 執行每日贈禮任務（每天自動執行1次）
result = buy_gift_for_friend_daily(d, device_id, times=1)

### 進階用法：檢查任務狀態

In [ ]:
# 查看任務執行記錄
from json_manager import JsonDataManager

device_id = 'emulator-5554'
manager = JsonDataManager(device_id, file_suffix="gift_task")

# 獲取任務記錄
record = manager.get_record("daily_gift_friend")

if record:
    print("任務執行記錄:")
    print(f"  執行時間: {record.get('datetime', 'Unknown')}")
    print(f"  狀態: {record.get('status', 'Unknown')}")
    print(f"  總次數: {record.get('total_count', 0)}")
    print(f"  成功次數: {record.get('success_count', 0)}")
    print(f"  失敗次數: {record.get('fail_count', 0)}")
    
    # 檢查是否為今天執行
    if manager.is_same_day("daily_gift_friend"):
        print("\n✓ 今天已執行過此任務")
    else:
        print("\n✗ 今天尚未執行此任務")
else:
    print("尚無任務執行記錄")

### 手動重置任務（用於測試或強制重新執行）

In [ ]:
# 手動刪除任務記錄（強制重新執行）
from json_manager import JsonDataManager

device_id = 'emulator-5554'
manager = JsonDataManager(device_id, file_suffix="gift_task")

# 載入數據
data = manager.load_data()

# 刪除特定任務記錄
if "daily_gift_friend" in data:
    del data["daily_gift_friend"]
    manager.save_data(data)
    print("✓ 已刪除任務記錄，可以重新執行")
else:
    print("✗ 找不到任務記錄")

# 或者完全清空所有記錄
# manager.save_data({})
# print("✓ 已清空所有任務記錄")

## 功能說明

### ✨ 主要特點

1. **自動每日檢查**
   - 使用 `json_manager` 記錄執行時間
   - 自動判斷今天是否已執行
   - 避免重複執行

2. **詳細執行統計**
   - 記錄總執行次數
   - 記錄成功/失敗次數
   - 保存執行時間戳

3. **靈活配置**
   - 可自定義執行次數（默認10次）
   - 可指定設備IP
   - 支援多設備管理

### 📊 JSON 記錄格式

```json
{
  "daily_gift_friend": {
    "timestamp": 1733654400.123,
    "date": "2025-12-08",
    "datetime": "2025-12-08 10:30:00",
    "status": "completed",
    "total_count": 10,
    "success_count": 9,
    "fail_count": 1
  }
}
```

### 🔄 使用流程

```python
# 1. 首次執行 - 會執行任務
result = buy_gift_for_friend_daily(d, 'emulator-5554', times=10)
# 輸出: 開始執行每日贈禮任務...

# 2. 同一天再次執行 - 會跳過
result = buy_gift_for_friend_daily(d, 'emulator-5554', times=10)
# 輸出: ✓ 今日贈禮任務已完成

# 3. 第二天執行 - 會重新執行
result = buy_gift_for_friend_daily(d, 'emulator-5554', times=10)
# 輸出: 開始執行每日贈禮任務...
```

In [ ]:
import new_cnn.cnn_model as cnn_model
import easyocr
from json_manager import ParkMarketDataManager
import BUY
device_id = 'emulator-5554'
Cnn_model = cnn_model.load_cnn_model("cnn_model.pth")
def goto_park(device,Cnn_model=Cnn_model):
    """停車主流程 main"""
    device.click(321, 913)
    while(1):
        cnn_result = cnn_model.predict_image(
            Cnn_model, device.screenshot(format='pillow'))
        if cnn_result == 'homeplace':
            break
    device.click(451, 451)
    time.sleep(1)
    

In [ ]:
# goto_park(d,Cnn_model=Cnn_model)
import img_tools
img = d.screenshot(format='opencv')
click_str_by_server(d,'坐騎改裝')



True

In [58]:
# 截圖並且顯示 可以點擊座標顯示該點的顏色值
img = d.screenshot(format='opencv')
def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        b, g, r = img[y, x]
        print(f"座標 ({x}, {y}) 的顏色值: R={r}, G={g}, B={b}")
cv2.imshow("Screenshot", img)
cv2.setMouseCallback("Screenshot", on_mouse)
cv2.waitKey(0)
cv2.destroyAllWindows()
# 沒有蒐藏的星星顏色
# 座標 (57, 197) 的顏色值: R=219, G=183, B=105
# 座標 (59, 191) 的顏色值: R=218, G=192, B=117
# 座標 (58, 195) 的顏色值: R=220, G=183, B=103
# 座標 (60, 196) 的顏色值: R=219, G=183, B=105
# 座標 (59, 196) 的顏色值: R=220, G=183, B=105
# 座標 (62, 194) 的顏色值: R=219, G=184, B=103
# 座標 (57, 188) 的顏色值: R=214, G=173, B=109
# 座標 (61, 190) 的顏色值: R=218, G=183, B=117
# 有蒐藏的星星顏色
# 座標 (57, 197) 的顏色值: R=245, G=219, B=36
# 座標 (59, 191) 的顏色值: R=248, G=241, B=75
# 座標 (58, 195) 的顏色值: R=241, G=218, B=42
# 座標 (60, 196) 的顏色值: R=225, G=191, B=31
# 座標 (59, 196) 的顏色值: R=239, G=209, B=33
# 座標 (62, 194) 的顏色值: R=236, G=199, B=48
# 座標 (57, 188) 的顏色值: R=249, G=217, B=114
# 座標 (61, 190) 的顏色值: R=247, G=212, B=68
# point_list =[(57, 197),(59, 191),(58, 195),(60, 196),(59, 196),(62, 194),(57, 188),(61, 190)]
# img = d.screenshot(format='opencv')
# print("顏色值列表:")
# for point in point_list:
#     b, g, r = img[point[1], point[0]]
#     print(f"座標 {point} 的顏色值: R={r}, G={g}, B={b}")

# 有蒐藏的星星顏色
# 座標 (57, 197) 的顏色值: R=245, G=219, B=36
# 座標 (59, 191) 的顏色值: R=248, G=241, B=75
# 座標 (58, 195) 的顏色值: R=241, G=218, B=42
# 座標 (60, 196) 的顏色值: R=225, G=191, B=31
# 座標 (59, 196) 的顏色值: R=239, G=209, B=33
# 座標 (62, 194) 的顏色值: R=236, G=199, B=48
# 座標 (57, 188) 的顏色值: R=249, G=217, B=114
# 座標 (61, 190) 的顏色值: R=247, G=212, B=68



座標 (235, 728) 的顏色值: R=81, G=156, B=221
座標 (234, 733) 的顏色值: R=69, G=132, B=199
座標 (245, 734) 的顏色值: R=69, G=148, B=217


In [ ]:


# print("rgbs:", rgbs)
# print("mode:", mode, "avg_score:", score)


In [ ]:
import cv2
import numpy as np
from typing import Iterable, Tuple, Literal
import time
import img_tools
Mode = Literal["collected", "not_collected"]
def star_score_rgb(rgb: Tuple[int, int, int]) -> float:
    r, g, b = rgb
    return (r + g) / 2.0 - b

def classify_star_rgb(
    rgbs: Iterable[Tuple[int, int, int]],
    threshold: float = 110.0
) -> tuple[Mode, float]:
    rgbs = list(rgbs)
    if not rgbs:
        raise ValueError("rgbs 不能是空的")
    scores = [star_score_rgb(rgb) for rgb in rgbs]
    avg = float(np.mean(scores))
    return ("collected" if avg >= threshold else "not_collected"), avg

def sample_rgbs_from_opencv_image(
    img_bgr: np.ndarray,
    points_xy: Iterable[Tuple[int, int]],
    clamp: bool = True
) -> list[Tuple[int, int, int]]:
    """
    img_bgr: OpenCV 圖 (H,W,3) BGR
    points_xy: (x,y) 座標（注意是 x,y）
    回傳: RGB list
    """
    h, w = img_bgr.shape[:2]
    rgbs = []
    for x, y in points_xy:
        if clamp:
            x = max(0, min(w - 1, int(x)))
            y = max(0, min(h - 1, int(y)))
        else:
            x, y = int(x), int(y)
            if not (0 <= x < w and 0 <= y < h):
                raise ValueError(f"point out of bounds: {(x,y)} (w={w}, h={h})")

        b, g, r = img_bgr[y, x]   # OpenCV 用 y,x 索引
        rgbs.append((int(r), int(g), int(b)))
    return rgbs

# === 你已經有這個 ===
def check_if_start(d):
    # === 用你提供的那些座標來取樣 ===
    points = [(57,197), (59,191), (58,195), (60,196), (59,196), (62,194), (57,188), (61,190)]

    img = d.screenshot(format='opencv')
    rgbs = sample_rgbs_from_opencv_image(img, points)
    mode, score = classify_star_rgb(rgbs)
    if mode == "collected":
        return True
    else:
        return False
def create_white_mask(img):
    hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # 定義白色範圍
    lower_blue = np.array([0, 0, 200])
    upper_blue = np.array([180, 30, 255])
    mask = cv2.inRange(hsv_img, lower_blue, upper_blue)
    return mask
def find_car(img):
    img1 = img[744:788]
    img = cv2.cvtColor(img1, cv2.COLOR_BGR2HSV)
    # mask = cv2.inRange(img, (37, 0, 0), (179, 255, 255))
    mask = cv2.inRange(img, (0, 101, 114), (179, 255, 255))
    # 膨脹
    mask = cv2.dilate(mask, None, iterations=2)
    # 侵蝕
    mask = cv2.erode(mask, None, iterations=1)
    # 計算輪廓
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # 顯示偵測到的遮罩與原始區域以便除錯（非阻塞）
    # try:
    #     cv2.imshow("find_car_bgr", img1)
    #     cv2.imshow("find_car_mask", mask)
    #     cv2.waitKey(0)
    # except Exception:
    #     # 在某些 headless 或非 GUI 環境下，imshow 會失敗；忽略該錯誤以維持流程
    #     pass
    contours = [cv2.boundingRect(
        contour) for contour in contours if cv2.contourArea(contour) > 1000]
    return contours
# img[719:741,:]
def canel_collect(d):
    img = d.screenshot(format='opencv')
    contours = find_car(img)
    print("找到車輛數量:", len(contours))
    for (x, y, w, h) in contours:
        print(f"車輛位置: x={x}, y={y}, w={w}, h={h}")
        cv2.rectangle(img, (x, y+720+2), (x + w, y + h+744), (0, 255, 0), 2)
        #傳給img_tools 判斷文字
        roi = img[y+720:y + h+700, x+w-20:x + w]
        # cv2.imshow("ROI", roi)
        # cv2.waitKey(0)
        mask = create_white_mask(roi)
        white_area = cv2.countNonZero(mask)
        if white_area >30:
            print("有停車")
            d.click(x + w - 10, y + 720 + (h // 2))
            time.sleep(0.3)
            img = d.screenshot(format='opencv') 
            points = [(57,197), (59,191), (58,195), (60,196), (59,196), (62,194), (57,188), (61,190)]
            rgbs = sample_rgbs_from_opencv_image(img, points)
            mode, score = classify_star_rgb(rgbs)   
            print("rgbs:", rgbs)
            print("mode:", mode)
            if mode == 'collected':
                print("已收藏")
                d.click(59,196)
            else:
                pass
def car_name(d):
    img = d.screenshot(format='opencv')[132:165,39:171]
    result = img_tools.analyze_skill_via_http(img)
    if result.get('success') != True:
        print("OCR服務無法連接")
    else:
        # print("OCR結果:")
        for item in result['ocr_results']:
            print(item['text'])
            return item['text']
    return ''
def park_time(d):
    img = d.screenshot(format='opencv')
    img = img[495:530,:]
    result = img_tools.analyze_skill_via_http(img)
    if result.get('success') != True:
        print("OCR服務無法連接")
    else:
        print("OCR結果:")
        for item in result['ocr_results']:
            print(item['text'])
            #今日累計停車時間246分鐘
            # 取出數字
            if '今日累計停車時間' in item['text']:
                minutes = 0
                text = item['text']
                minutes = ''.join(filter(str.isdigit, text))
                print("今日累計停車時間:", minutes, "分鐘")
            return int(minutes)
    return 0

def choose_car(d,target_name_list=[]):
    collect_car_num =0
    start_time = time.time()
    while collect_car_num <= 5 or (time.time() - start_time)<60:
        img = d.screenshot(format='opencv')
        contours = find_car(img)
        print("找到車輛數量:", len(contours))
        #使用x 大小排序 由小到大
        contours = sorted(contours, key=lambda x: x[2])
        for (x, y, w, h) in contours:
            print(f"車輛位置: x={x}, y={y}, w={w}, h={h}")
            cv2.rectangle(img, (x, y+720), (x + w, y + h+744), (0, 255, 0), 2)
            #傳給img_tools 判斷文字
            roi = img[y+720:y + h+700, x+w-20:x + w]
            cv2.imshow("ROI", roi)
            cv2.waitKey(0)
            mask = create_blue_mask(roi)
            blue_area = cv2.countNonZero(mask)
            print("藍色區域:", blue_area)
            if blue_area > 100:
                #沒有停車
                continue
            print("沒停車")
            if check_if_start(d):
                print("已收藏")
                continue
            d.click(x + w - 10, y + 720 + (h // 2))
            time.sleep(0.3)
            if int(park_time(d))>=240:
                print("停車時間已滿")
                continue
            else:
                print("選擇此車停車")
                name = car_name(d)
                if name in target_name_list:
                    print("目標車輛，選擇停車")
                    collect_car_num +=1
                    d.click(62,198)
        if collect_car_num >=5:
            break
        try:
            d.swipe(414,720,80,720)
            d.click(90,758)
            time.sleep(1)
        except Exception as e:
            print(f"滑動或點擊時發生錯誤: {e}")
            time.sleep(2)
    img_tools.click_str_by_server(d,'坐騎改裝',y_range=(0,100),shift_y=885-84)
    time.sleep(3)
def new_park_way(d):
    img_tools.click_str_by_server(d,'坐騎改裝')
    time.sleep(2)
    canel_collect(d)     
    time.sleep(1)
    #凌晨12點刷新 重新蒐藏要停的車
    img_tools.click_str_by_server(d,'坐騎改裝',y_range=(0,100),shift_y=885-84)
    time.sleep(2)
    img_tools.click_str_by_server(d,'坐騎改裝')
    time.sleep(1)
    return choose_car(d)
d= u2.connect('emulator-5558')
stop = False
def if_already_parked(img, x, y, w, h):

    roi = img[y+720:y + h+700, x+w-20:x + w]
    # cv2.imshow("ROI", roi)
    # cv2.waitKey(0)
    mask = create_white_mask(roi)
    white_area = cv2.countNonZero(mask)

    if white_area >30:
        print("有停車")
        return True
    else:
        print("沒停車")
        return False
    
list_target = ['銀時的摩托車','遙遙领先','雲飘飘兮', '圆圆蛙','青牛哞哞' ,'白虎' ,'極光戰龍' ,'七彩祥雲'  ,'来潜水鸭' ,'堕落摩托' ,'格拉尼' ,'風火輪' ,'蓮花寶座' ]
# canel_collect(d)


In [ ]:
import time
import uiautomator2 as u2
import os
import cv2
import numpy as np
from typing import Iterable, Tuple
import random
import img_tools
from typing_extensions import Literal
Mode = Literal["collected", "not_collected"]
def sample_rgbs_from_opencv_image(
    img_bgr: np.ndarray,
    points_xy: Iterable[Tuple[int, int]],
    clamp: bool = True
) -> list[Tuple[int, int, int]]:
    """
    img_bgr: OpenCV 圖 (H,W,3) BGR
    points_xy: (x,y) 座標（注意是 x,y）
    回傳: RGB list
    """
    h, w = img_bgr.shape[:2]
    rgbs = []
    for x, y in points_xy:
        if clamp:
            x = max(0, min(w - 1, int(x)))
            y = max(0, min(h - 1, int(y)))
        else:
            x, y = int(x), int(y)
            if not (0 <= x < w and 0 <= y < h):
                raise ValueError(f"point out of bounds: {(x,y)} (w={w}, h={h})")

        b, g, r = img_bgr[y, x]   # OpenCV 用 y,x 索引
        rgbs.append((int(r), int(g), int(b)))
    return rgbs
def find_car(img):
    img1 = img[744:788]
    img = cv2.cvtColor(img1, cv2.COLOR_BGR2HSV)
    # mask = cv2.inRange(img, (37, 0, 0), (179, 255, 255))
    mask = cv2.inRange(img, (0, 101, 114), (179, 255, 255))
    # 膨脹
    mask = cv2.dilate(mask, None, iterations=2)
    # 侵蝕
    mask = cv2.erode(mask, None, iterations=1)
    # 計算輪廓
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # 顯示偵測到的遮罩與原始區域以便除錯（非阻塞）
    # try:
    #     cv2.imshow("find_car_bgr", img1)
    #     cv2.imshow("find_car_mask", mask)
    #     cv2.waitKey(0)
    # except Exception:
    #     # 在某些 headless 或非 GUI 環境下，imshow 會失敗；忽略該錯誤以維持流程
    #     pass
    contours = [cv2.boundingRect(
        contour) for contour in contours if cv2.contourArea(contour) > 1000]
    return contours
def star_score_rgb(rgb: Tuple[int, int, int]) -> float:
    r, g, b = rgb
    return (r + g) / 2.0 - b
def classify_star_rgb(
    rgbs: Iterable[Tuple[int, int, int]],
    threshold: float = 110.0
) -> tuple[Mode, float]:
    rgbs = list(rgbs)
    if not rgbs:
        raise ValueError("rgbs 不能是空的")
    scores = [star_score_rgb(rgb) for rgb in rgbs]
    avg = float(np.mean(scores))
    return ("collected" if avg >= threshold else "not_collected"), avg
def car_name(d):
    img = d.screenshot(format='opencv')[132:165,39:171]
    result = img_tools.analyze_skill_via_http(img)
    if result.get('success') != True:
        print("OCR服務無法連接")
    else:
        # print("OCR結果:")
        for item in result['ocr_results']:
            print(item['text'])
            return item['text']
    return ''
def create_white_mask(img):
    hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # 定義白色範圍
    lower_blue = np.array([0, 0, 200])
    upper_blue = np.array([180, 30, 255])
    mask = cv2.inRange(hsv_img, lower_blue, upper_blue)
    return mask
def if_already_parked(img, x, y, w, h):

    roi = img[y+720:y + h+700, x+w-20:x + w]
    # cv2.imshow("ROI", roi)
    # cv2.waitKey(0)
    mask = create_white_mask(roi)
    white_area = cv2.countNonZero(mask)

    if white_area >30:
        print("有停車")
        return True
    else:
        print("沒停車")
        return False
def check_if_start(d):
    # === 用你提供的那些座標來取樣 ===
    points = [(57,197), (59,191), (58,195), (60,196), (59,196), (62,194), (57,188), (61,190)]

    img = d.screenshot(format='opencv')
    rgbs = sample_rgbs_from_opencv_image(img, points)
    mode, score = classify_star_rgb(rgbs)
    if mode == "collected":
        return True
    else:
        return False
def park_time(d):
    img = d.screenshot(format='opencv')
    img = img[495:530,:]
    result = img_tools.analyze_skill_via_http(img)
    if result.get('success') != True:
        print("OCR服務無法連接")
    else:
        print("OCR結果:")
        for item in result['ocr_results']:
            print(item['text'])
            #今日累計停車時間246分鐘
            # 取出數字
            if '今日累計停車時間' in item['text']:
                minutes = 0
                text = item['text']
                minutes = ''.join(filter(str.isdigit, text))
                print("今日累計停車時間:", minutes, "分鐘")
            return int(minutes)
    return 0
def park_and_start(d,use_target=False, list_target=[],stop_car_name='螺旋小飛機'):
    stop = False
    while True:
        img = d.screenshot(format='opencv')
        contours = find_car(img)
        #排序 按照x座標由小到大
        contours = sorted(contours, key=lambda x: x[0])
        if len(contours) ==0:
            print("沒有車輛，等待3秒")
            time.sleep(3)
            continue
        for (x, y, w, h) in contours:
            print(f"車輛位置: x={x}, y={y}, w={w}, h={h}")
            # cv2.rectangle(img, (x, y+720), (x + w, y + h+744), (0, 255, 0), 2)
            d.click(x + w - 10, y + 720 + (h // 2))
            time.sleep(0.5)
            img = d.screenshot(format='opencv')
            if_collect = check_if_start(d)
            if if_already_parked(img, x, y, w, h) and if_collect:
                #取消蒐藏
                print("已停車且已收藏，取消蒐藏")
                d.click(59,196)
                time.sleep(0.5)
                continue
            if if_already_parked(img, x, y, w, h):
                print("已停車")
                continue
            elif if_collect: #已經蒐藏還沒停車 
                print("已收藏")
                continue
            print('car_name:',car_name(d))
            if stop_car_name !='' and car_name(d) == stop_car_name:
                print("到達停止車輛，結束程式")
                stop = True
                break
            if car_name(d) not in list_target and use_target:
                print("非目標車輛，跳過")
                continue
            if park_time(d)>=240:
                pass
            else:
                print("可選為停車車輛 加入蒐藏")
                d.click(62,198)
                time.sleep(0.5)
                if check_if_start(d):
                    print("已收藏")
                else:
                    print("收藏失敗")
                    stop = True
                    break
                continue
        if stop:
            break
        d.swipe(414,720,50,720)
        d.click(67,764)
        time.sleep(0.5)

def new_park_way(d,ip):
    img_tools.click_str_by_server(d,'坐騎改裝',y_range=(741,843))
    time.sleep(2)
    if 'emulator-5558' in ip:
        list_target = ['銀時的摩托車','遙遙领先','雲飘飘兮', '圆圆蛙','青牛哞哞' ,'白虎' ,'極光戰龍' ,'七彩祥雲'  ,'来潜水鸭' ,'堕落摩托' ,'格拉尼' ,'風火輪' ,'蓮花寶座','朗姆酒桶','自助小摩托','青羽']
        park_and_start (d,use_target=True,list_target=list_target,stop_car_name='紫金葫蘆')
    else:
        list_target = []
        park_and_start (d,use_target=False,list_target=list_target,stop_car_name='紫金葫蘆')
    time.sleep(1)
    #凌晨12點刷新 重新蒐藏要停的車
    img_tools.click_str_by_server(d,'坐騎改裝',y_range=(0,100),shift_y=885-84)
    return True
new_park_way(d,'emulator-5558')


車輛位置: x=35, y=0, w=73, h=44
沒停車
沒停車
焚焰魔狼
car_name: 焚焰魔狼
焚焰魔狼
焚焰魔狼
非目標車輛，跳過
車輛位置: x=113, y=0, w=66, h=44
有停車
有停車
已停車
車輛位置: x=188, y=0, w=65, h=44
沒停車
沒停車
獵豹零號機
car_name: 獵豹零號機
獵豹零號機
獵豹零號機
非目標車輛，跳過
車輛位置: x=263, y=0, w=66, h=44
有停車
有停車
已停車
車輛位置: x=338, y=0, w=66, h=44
有停車
有停車
已停車
車輛位置: x=413, y=0, w=66, h=44
有停車
有停車
已停車
車輛位置: x=35, y=0, w=73, h=44
有停車
有停車
已停車
車輛位置: x=113, y=0, w=66, h=44
沒停車
沒停車
烈焰嘯天
car_name: 烈焰嘯天
烈焰啸天
烈焰啸天
非目標車輛，跳過
車輛位置: x=188, y=0, w=66, h=44
沒停車
沒停車
常夜魚燈
car_name: 常夜魚燈
常夜魚燈
常夜魚燈
非目標車輛，跳過
車輛位置: x=263, y=0, w=66, h=44
沒停車
沒停車
千羽靈凰
car_name: 千羽靈凰
千羽靈凰
千羽靈凰
非目標車輛，跳過
車輛位置: x=338, y=0, w=66, h=44
沒停車
沒停車
亡靈飛車
car_name: 亡靈飛車
亡靈飛車
亡靈飛車
非目標車輛，跳過
車輛位置: x=413, y=0, w=66, h=44
沒停車
沒停車
雲間客
car_name: 雲間客
雲間客
雲間客
非目標車輛，跳過
車輛位置: x=43, y=0, w=73, h=44
沒停車
沒停車
雲間客
car_name: 雲間客
雲間客
雲間客
非目標車輛，跳過
車輛位置: x=121, y=0, w=66, h=44
沒停車
沒停車
狂浪巡行者
car_name: 狂浪巡行者
狂浪巡行者
狂浪巡行者
非目標車輛，跳過
車輛位置: x=197, y=0, w=66, h=44
沒停車
沒停車
銀時的摩托車
car_name: 銀時的摩托車
銀時的摩托車
銀時的摩托車
OCR結果:
今日累計停車時間0分鐘
今日累計停車時間: 0 分鐘
可選為停車

True

In [ ]:
def goto_home(device):        
    device.click(470, 922)
    time.sleep(2)
    device.click(321, 913)
    time.sleep(3)
goto_home(d)

: 

In [ ]:
import easyocr 
easyocr_reader = easyocr.Reader(['ch_tra','en'])
import uiautomator2 as u2
import numpy as np
import time
import img_tools
import cv2
import os
from tools import click_white
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)
def stage_by_str(d, ocr_str: list, img: np.ndarray) -> str:
    """
    Determines the current game stage based on OCR text and saves a debug image.

    Args:
        d: The uiautomator2 device object (remains for potential future use).
        ocr_str: A list of strings detected by OCR.
        img: The OpenCV image corresponding to the OCR text.

    Returns:
        The name of the detected stage or "未知".
    """
    # Combine list of strings into a single string for easier searching
    full_text = "".join(ocr_str)
    print("OCR偵測文字:", full_text)
    # Mapping of keywords to stage names and their properties
    stage_keywords = {
        "車位倉庫": "車位倉庫",
        "購物管家":"購物管家",
        "你的帳號在另一個地方登錄": "異地登錄",
        "退出遊戲": "異地登錄",
        "公告": "公告",
        "方案": "主頁面",
        "放置獎勵": "放置獎勵",
        "離線獎勵": "放置獎勵",
        "家族商店": "家族",
        "家族亂鬥": "家族",
        "征戰熔岩巨獸": "征戰熔岩巨獸",
        
    }

    for keyword, stage_name in stage_keywords.items():
        # Special condition for "征戰熔岩巨獸"
        if  "購物管家" in full_text and "離線獎勵" in full_text:
            return "放置獎勵"
        if  "購物管家" in full_text and "車位倉庫" in full_text:
            return "車位倉庫"
        if stage_name == "征戰熔岩巨獸":
            if "征戰熔岩巨獸" in full_text and "掃蕩" in full_text:
                # img_tools.save_stage_debug_image(stage_name, img)
                return stage_name
        # General condition for other stages
        elif keyword in full_text:
            # img_tools.save_stage_debug_image(stage_name, img)
            return stage_name
            
    return "未知"
def reward(d, easyocr_reader):
    d.click(162, 725)
    time.sleep(5)
    img = d.screenshot(format='opencv')
    result = easyocr_reader.readtext(img, detail=0)
    if "領取" in result or "放置獎勵" in result:
        logger.info(img[328, 135])
        if abs(np.sum(img[328, 135])-np.sum([206, 237, 247])) > 12:
            # if not os.path.exists("reward_get"):
            #     os.makedirs("reward_get")
            # cv2.imwrite(
            #     "reward_get/reward_get_{}.jpg".format(time.time()), img)
            click_white(d)
            time.sleep(1)
        d.click(330, 725)
        time.sleep(5)
        click_white(d)
        time.sleep(1)
    click_white(d)
d = u2.connect('emulator-5554')         
wait_time = time.time()       
while (1):
    img = d.screenshot(format='opencv')
    ocr_result = easyocr_reader.readtext(img, detail=0)
    current_stage = stage_by_str(d, ocr_result, img)
    if current_stage == "離線獎勵" or current_stage == "放置獎勵":
        reward(d, easyocr_reader)
    if "公告" in ocr_result:
        d.click(248, 812)
        time.sleep(1)
        click_white(d)
        time.sleep(1)
    if current_stage == "購物管家":
        logger.info("購物管家頁面，點擊返回主頁面")
        img_tools.click_str_by_server(d, '採購', y_range=(690,740))
        time.sleep(2)
        click_white(d)
        img_tools.click_str_by_server(d, '副本管家', y_range=(773,839))
        time.sleep(2)
        img_tools.click_str_by_server(d, '掃蕩', y_range=(690,740))
        time.sleep(2)
        for i in range(6):
            click_white(d)
        time.sleep(2)
        break
    if current_stage == "主頁面":
        break 
    if current_stage =="車位倉庫":
        logger.info("車位倉庫頁面，點擊返回主頁面")
        img_tools.click_str_by_server(d, '領取', y_range=(697,737))
        time.sleep(2)
        click_white(d)
        time.sleep(1)
    if current_stage == "異地登錄":
        logger.info("異地登錄頁面，重新啟動遊戲")
        d.app_stop("com.mxdzz.tw.and")
        time.sleep(1)
        # 重新啟動時使用圖示點擊
        start_game_by_icon(d, ip)
        time.sleep(30+random.randint(0, 5))
        wait_time = time.time()
    if current_stage == "未知":
        logger.info(f"[{ip}] 未知頁面，等待中...")
        d.press("back")
        time.sleep(5)    
    if time.time()-wait_time > 60:
        d.app_stop("com.mxdzz.tw.and")
        time.sleep(1)
        # 重新啟動時使用圖示點擊
        logger.info(f"[{ip}] 等待超時,重新啟動遊戲")
        # start_game_by_icon(d, ip)
        time.sleep(30+random.randint(0, 5))
        wait_time = time.time()


INFO:__main__:購物管家頁面，點擊返回主頁面


OCR偵測文字: P: 0/10.0d/: 0.0Xw: O.0Yw: O.0Prs:1.0Size305=83561191148828 1119072購物管家特權卡領取每日好禮X150已領取X600X200已領取商店自動購買Oll設置購物車家族商店榮罹商店積分商店車友商行星將商店神秘商人召喚商店採購購物管家副本管家Sii0p角色同伴副本家族商店
